In [ ]:
# ============================================================
# 0. SETUP + LOAD DATA + COLOR CONFIG
# ============================================================

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", font_scale=1.0)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

REQUIRED_FILES = [
    "products.csv",
    "customers.csv",
    "promotions.csv",
    "geography.csv",
    "orders.csv",
    "order_items.csv",
    "payments.csv",
    "shipments.csv",
    "returns.csv",
    "reviews.csv",
    "sales.csv",
    "sample_submission.csv",
    "inventory.csv",
    "web_traffic.csv",
]

DATA_DIRS = [
    Path("/content"),
    Path("/mnt/data"),
    Path("."),
]

def find_file(filename, search_dirs=DATA_DIRS):
    for d in search_dirs:
        p = d / filename
        if p.exists():
            return p
    return None

missing_files = [f for f in REQUIRED_FILES if find_file(f) is None]

if missing_files:
    print("Các file sau chưa có trong môi trường Colab:")
    print(missing_files)
    print("Vui lòng upload các file này. Không cần sửa code.")
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as e:
        raise FileNotFoundError(
            f"Không tìm thấy file: {missing_files}. "
            "Nếu không chạy trên Colab, hãy đặt các file CSV cùng thư mục với notebook."
        )

file_paths = {f: find_file(f) for f in REQUIRED_FILES}
still_missing = [f for f, p in file_paths.items() if p is None]

if still_missing:
    raise FileNotFoundError(f"Vẫn còn thiếu file: {still_missing}")

print("Đã tìm thấy đầy đủ file CSV.")
for f, p in file_paths.items():
    print(f"{f}: {p}")

products = pd.read_csv(file_paths["products.csv"])
customers = pd.read_csv(file_paths["customers.csv"], parse_dates=["signup_date"])
promotions = pd.read_csv(
    file_paths["promotions.csv"],
    parse_dates=["start_date", "end_date"],
    dtype={"promo_id": "string"}
)
geography = pd.read_csv(file_paths["geography.csv"])

orders = pd.read_csv(file_paths["orders.csv"], parse_dates=["order_date"])
order_items = pd.read_csv(
    file_paths["order_items.csv"],
    dtype={"promo_id": "string", "promo_id_2": "string"},
    low_memory=False
)

payments = pd.read_csv(file_paths["payments.csv"])
shipments = pd.read_csv(file_paths["shipments.csv"], parse_dates=["ship_date", "delivery_date"])
returns = pd.read_csv(file_paths["returns.csv"], parse_dates=["return_date"])
reviews = pd.read_csv(file_paths["reviews.csv"], parse_dates=["review_date"])

sales = pd.read_csv(file_paths["sales.csv"], parse_dates=["Date"])
sample_submission = pd.read_csv(file_paths["sample_submission.csv"], parse_dates=["Date"])
inventory = pd.read_csv(file_paths["inventory.csv"], parse_dates=["snapshot_date"])
web_traffic = pd.read_csv(file_paths["web_traffic.csv"], parse_dates=["date"])

REVENUE_SCALE = 1_000_000
REVENUE_UNIT = "million"

BASE_CATEGORY_COLORS = {
    "Streetwear": "#1f77b4",
    "Outdoor": "#ff7f0e",
    "Casual": "#2ca02c",
    "GenZ": "#d62728",
}

category_values = sorted(products["category"].dropna().unique())
fallback_colors = sns.color_palette("tab10", n_colors=max(len(category_values), 4))

category_palette = {
    cat: BASE_CATEGORY_COLORS.get(cat, fallback_colors[i])
    for i, cat in enumerate(category_values)
}

metric_palette = {
    "Revenue": "#1f77b4",
    "Orders": "#ff7f0e",
    "AOV": "#2ca02c",
    "Active_Customers": "#9467bd",
    "ARPU": "#d62728",
    "Gross_Margin_Pct": "#8c564b",
}

month_order = list(range(1, 13))
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

def money_axis(x, pos):
    return f"{x / REVENUE_SCALE:,.0f}"

def pct_axis(x, pos):
    return f"{x:.0%}"

def safe_divide(a, b):
    return np.where(b != 0, a / b, np.nan)

def display_table(df, money_cols=None, pct_cols=None, int_cols=None):
    money_cols = money_cols or []
    pct_cols = pct_cols or []
    int_cols = int_cols or []

    fmt = {}

    for c in money_cols:
        if c in df.columns:
            fmt[c] = "{:,.2f}"

    for c in pct_cols:
        if c in df.columns:
            fmt[c] = "{:.2%}"

    for c in int_cols:
        if c in df.columns:
            fmt[c] = "{:,.0f}"

    display(df.style.format(fmt))

print("\nRandom seed:", SEED)
print("Category colors:")
for k, v in category_palette.items():
    print(f"{k}: {v}")

Các file sau chưa có trong môi trường Colab:
['products.csv', 'customers.csv', 'promotions.csv', 'geography.csv', 'orders.csv', 'order_items.csv', 'payments.csv', 'shipments.csv', 'returns.csv', 'reviews.csv', 'sales.csv', 'sample_submission.csv', 'inventory.csv', 'web_traffic.csv']
Vui lòng upload các file này. Không cần sửa code.


In [ ]:
# ============================================================
# 1. BUILD FACT TABLE + DAILY/MONTHLY/QUARTERLY/YEARLY TABLES
# ============================================================

product_cols = [
    "product_id", "product_name", "category", "segment",
    "size", "color", "price", "cogs"
]

order_cols = [
    "order_id", "order_date", "customer_id", "zip",
    "order_status", "payment_method", "device_type", "order_source"
]

fact = (
    order_items
    .merge(products[product_cols], on="product_id", how="left")
    .merge(orders[order_cols], on="order_id", how="left")
)

fact["date"] = fact["order_date"].dt.normalize()

numeric_cols = ["quantity", "unit_price", "discount_amount", "price", "cogs"]
for c in numeric_cols:
    fact[c] = pd.to_numeric(fact[c], errors="coerce")

fact["discount_amount"] = fact["discount_amount"].fillna(0)

fact["item_revenue"] = fact["quantity"] * fact["unit_price"]
fact["revenue_before_discount"] = fact["item_revenue"] + fact["discount_amount"]
fact["cogs_total"] = fact["quantity"] * fact["cogs"]
fact["gross_profit"] = fact["item_revenue"] - fact["cogs_total"]

fact["gross_margin_pct"] = safe_divide(fact["gross_profit"], fact["item_revenue"])
fact["discount_rate"] = safe_divide(fact["discount_amount"], fact["revenue_before_discount"])

def aggregate_revenue(df, freq):
    out = (
        df.groupby(pd.Grouper(key="date", freq=freq))
        .agg(
            Revenue=("item_revenue", "sum"),
            COGS=("cogs_total", "sum"),
            Gross_Profit=("gross_profit", "sum"),
            Revenue_Before_Discount=("revenue_before_discount", "sum"),
            Discount=("discount_amount", "sum"),
            Units=("quantity", "sum"),
            Orders=("order_id", "nunique"),
            Active_Customers=("customer_id", "nunique"),
        )
        .reset_index()
    )

    out["AOV"] = safe_divide(out["Revenue"], out["Orders"])
    out["ARPU"] = safe_divide(out["Revenue"], out["Active_Customers"])
    out["Gross_Margin_Pct"] = safe_divide(out["Gross_Profit"], out["Revenue"])
    out["Discount_Rate"] = safe_divide(out["Discount"], out["Revenue_Before_Discount"])

    return out

daily = aggregate_revenue(fact, "D")

full_daily_index = pd.date_range(
    start=daily["date"].min(),
    end=daily["date"].max(),
    freq="D"
)

daily = (
    daily.set_index("date")
    .reindex(full_daily_index)
    .rename_axis("date")
    .reset_index()
)

fill_cols = [
    "Revenue", "COGS", "Gross_Profit", "Revenue_Before_Discount",
    "Discount", "Units", "Orders", "Active_Customers"
]

daily[fill_cols] = daily[fill_cols].fillna(0)
daily["AOV"] = safe_divide(daily["Revenue"], daily["Orders"])
daily["ARPU"] = safe_divide(daily["Revenue"], daily["Active_Customers"])
daily["Gross_Margin_Pct"] = safe_divide(daily["Gross_Profit"], daily["Revenue"])
daily["Discount_Rate"] = safe_divide(daily["Discount"], daily["Revenue_Before_Discount"])

daily["year"] = daily["date"].dt.year
daily["quarter"] = daily["date"].dt.to_period("Q").astype(str)
daily["month"] = daily["date"].dt.month
daily["month_name"] = daily["date"].dt.month_name()
daily["weekday"] = daily["date"].dt.day_name()
daily["weekday_num"] = daily["date"].dt.weekday

monthly = aggregate_revenue(fact, "MS")
monthly["year"] = monthly["date"].dt.year
monthly["month"] = monthly["date"].dt.month
monthly["month_name"] = monthly["date"].dt.month_name()
monthly["period"] = monthly["date"].dt.to_period("M").astype(str)

quarterly = aggregate_revenue(fact, "QS")
quarterly["year"] = quarterly["date"].dt.year
quarterly["quarter_num"] = quarterly["date"].dt.quarter
quarterly["period"] = quarterly["date"].dt.to_period("Q").astype(str)

yearly = aggregate_revenue(fact, "YS")
yearly["year"] = yearly["date"].dt.year

monthly["YoY_Revenue_Growth"] = monthly["Revenue"].pct_change(12)
monthly["YoY_Orders_Growth"] = monthly["Orders"].pct_change(12)
monthly["YoY_AOV_Growth"] = monthly["AOV"].pct_change(12)
monthly["YoY_Active_Customers_Growth"] = monthly["Active_Customers"].pct_change(12)
monthly["YoY_ARPU_Growth"] = monthly["ARPU"].pct_change(12)

quarterly["YoY_Revenue_Growth"] = quarterly["Revenue"].pct_change(4)
quarterly["YoY_Orders_Growth"] = quarterly["Orders"].pct_change(4)
quarterly["YoY_AOV_Growth"] = quarterly["AOV"].pct_change(4)

yearly["YoY_Revenue_Growth"] = yearly["Revenue"].pct_change()
yearly["YoY_Orders_Growth"] = yearly["Orders"].pct_change()
yearly["YoY_AOV_Growth"] = yearly["AOV"].pct_change()
yearly["YoY_Active_Customers_Growth"] = yearly["Active_Customers"].pct_change()
yearly["YoY_ARPU_Growth"] = yearly["ARPU"].pct_change()

data_summary = pd.DataFrame({
    "table": ["fact", "daily", "monthly", "quarterly", "yearly"],
    "rows": [len(fact), len(daily), len(monthly), len(quarterly), len(yearly)],
    "date_min": [
        fact["date"].min(), daily["date"].min(), monthly["date"].min(),
        quarterly["date"].min(), yearly["date"].min()
    ],
    "date_max": [
        fact["date"].max(), daily["date"].max(), monthly["date"].max(),
        quarterly["date"].max(), yearly["date"].max()
    ],
})

display(data_summary)
display(fact.head())
display(daily.head())

In [ ]:
# ============================================================
# A1. REVENUE SUMMARY TABLES
# ============================================================

yearly_report = yearly[[
    "year", "Revenue", "COGS", "Gross_Profit", "Gross_Margin_Pct",
    "Orders", "AOV", "Active_Customers", "ARPU",
    "YoY_Revenue_Growth", "YoY_Orders_Growth", "YoY_AOV_Growth",
    "YoY_Active_Customers_Growth", "YoY_ARPU_Growth"
]].copy()

quarterly_report = quarterly[[
    "period", "Revenue", "COGS", "Gross_Profit", "Gross_Margin_Pct",
    "Orders", "AOV", "Active_Customers", "ARPU",
    "YoY_Revenue_Growth", "YoY_Orders_Growth", "YoY_AOV_Growth"
]].copy()

monthly_report = monthly[[
    "period", "year", "month", "month_name",
    "Revenue", "COGS", "Gross_Profit", "Gross_Margin_Pct",
    "Orders", "AOV", "Active_Customers", "ARPU",
    "YoY_Revenue_Growth", "YoY_Orders_Growth", "YoY_AOV_Growth",
    "YoY_Active_Customers_Growth", "YoY_ARPU_Growth"
]].copy()

print("YEARLY SUMMARY")
display_table(
    yearly_report,
    money_cols=["Revenue", "COGS", "Gross_Profit", "AOV", "ARPU"],
    pct_cols=[
        "Gross_Margin_Pct", "YoY_Revenue_Growth", "YoY_Orders_Growth",
        "YoY_AOV_Growth", "YoY_Active_Customers_Growth", "YoY_ARPU_Growth"
    ],
    int_cols=["Orders", "Active_Customers"]
)

print("QUARTERLY SUMMARY - LAST 20 QUARTERS")
display_table(
    quarterly_report.tail(20),
    money_cols=["Revenue", "COGS", "Gross_Profit", "AOV", "ARPU"],
    pct_cols=["Gross_Margin_Pct", "YoY_Revenue_Growth", "YoY_Orders_Growth", "YoY_AOV_Growth"],
    int_cols=["Orders", "Active_Customers"]
)

print("MONTHLY SUMMARY - LAST 24 MONTHS")
display_table(
    monthly_report.tail(24),
    money_cols=["Revenue", "COGS", "Gross_Profit", "AOV", "ARPU"],
    pct_cols=[
        "Gross_Margin_Pct", "YoY_Revenue_Growth", "YoY_Orders_Growth",
        "YoY_AOV_Growth", "YoY_Active_Customers_Growth", "YoY_ARPU_Growth"
    ],
    int_cols=["Orders", "Active_Customers"]
)

In [ ]:
# ============================================================
# A2. QUARTERLY REVENUE LINE CHART
# ============================================================

plot_df = quarterly.copy()
plot_df["Revenue_scaled"] = plot_df["Revenue"] / REVENUE_SCALE

plt.figure(figsize=(15, 6))

sns.lineplot(
    data=plot_df,
    x="date",
    y="Revenue_scaled",
    marker="o",
    linewidth=2.3,
    color=metric_palette["Revenue"],
)

plt.title("Quarterly Revenue Trend, 2012–2022", fontsize=16, fontweight="bold")
plt.xlabel("Quarter")
plt.ylabel(f"Revenue ({REVENUE_UNIT})")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# A3. MONTHLY SEASONALITY
# ============================================================

seasonality = (
    monthly
    .groupby("month", as_index=False)
    .agg(
        Avg_Revenue=("Revenue", "mean"),
        Median_Revenue=("Revenue", "median"),
        Avg_Orders=("Orders", "mean"),
        Avg_AOV=("AOV", "mean"),
        Avg_Active_Customers=("Active_Customers", "mean"),
        Avg_ARPU=("ARPU", "mean"),
    )
)

overall_avg_monthly_revenue = monthly["Revenue"].mean()

seasonality["Seasonality_Index"] = (
    seasonality["Avg_Revenue"] / overall_avg_monthly_revenue
)

seasonality["month_name"] = seasonality["month"].map(lambda m: month_labels[m - 1])

seasonality["Seasonality_Class"] = np.select(
    [
        seasonality["Seasonality_Index"] >= 1.10,
        seasonality["Seasonality_Index"] >= 1.03,
        seasonality["Seasonality_Index"] <= 0.90,
        seasonality["Seasonality_Index"] <= 0.97,
    ],
    [
        "Strong peak",
        "Mild peak",
        "Strong trough",
        "Mild trough",
    ],
    default="Normal",
)

seasonality = seasonality.sort_values("month")

display_table(
    seasonality,
    money_cols=["Avg_Revenue", "Median_Revenue", "Avg_AOV", "Avg_ARPU"],
    pct_cols=["Seasonality_Index"],
    int_cols=["Avg_Orders", "Avg_Active_Customers"],
)

plt.figure(figsize=(13, 6))

sns.barplot(
    data=seasonality,
    x="month_name",
    y="Avg_Revenue",
    color="#4C78A8",
)

plt.axhline(
    overall_avg_monthly_revenue,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Overall monthly average",
)

plt.title("Average Revenue by Month", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel(f"Average Revenue ({REVENUE_UNIT})")
plt.gca().yaxis.set_major_formatter(mtick.FuncFormatter(money_axis))
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))

sns.barplot(
    data=seasonality,
    x="month_name",
    y="Seasonality_Index",
    hue="Seasonality_Class",
    dodge=False,
    palette={
        "Strong peak": "#1f77b4",
        "Mild peak": "#6baed6",
        "Normal": "#bdbdbd",
        "Mild trough": "#fdae6b",
        "Strong trough": "#d62728",
    },
)

plt.axhline(1.0, color="black", linestyle="--", linewidth=1)
plt.title("Seasonality Index by Month", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Seasonality Index")
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.legend(title="Seasonality Class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# A4. INDEXED REVENUE / ORDERS / AOV
# ============================================================

indexed = quarterly[["date", "period", "Revenue", "Orders", "AOV"]].copy()
base = indexed[["Revenue", "Orders", "AOV"]].iloc[0]

for col in ["Revenue", "Orders", "AOV"]:
    indexed[f"{col}_Index"] = indexed[col] / base[col] * 100

indexed_long = indexed.melt(
    id_vars=["date", "period"],
    value_vars=["Revenue_Index", "Orders_Index", "AOV_Index"],
    var_name="Metric",
    value_name="Index",
)

indexed_long["Metric"] = indexed_long["Metric"].str.replace("_Index", "", regex=False)

plt.figure(figsize=(15, 6))

sns.lineplot(
    data=indexed_long,
    x="date",
    y="Index",
    hue="Metric",
    marker="o",
    linewidth=2.3,
    palette={k: metric_palette[k] for k in ["Revenue", "Orders", "AOV"]},
)

plt.axhline(100, color="black", linestyle="--", linewidth=1)
plt.title("Indexed Quarterly Revenue, Orders, and AOV", fontsize=16, fontweight="bold")
plt.xlabel("Quarter")
plt.ylabel("Index, first quarter = 100")
plt.legend(title="Metric")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# A5. YEAR x MONTH REVENUE HEATMAP
# ============================================================

heatmap_data = (
    monthly
    .assign(Revenue_scaled=lambda d: d["Revenue"] / REVENUE_SCALE)
    .pivot(index="year", columns="month", values="Revenue_scaled")
    .reindex(columns=month_order)
)

plt.figure(figsize=(14, 7))

sns.heatmap(
    heatmap_data,
    cmap="YlGnBu",
    linewidths=0.4,
    linecolor="white",
    annot=True,
    fmt=".0f",
    mask=heatmap_data.isna(),
    cbar_kws={"label": f"Revenue ({REVENUE_UNIT})"},
)

plt.title("Revenue Heatmap by Year and Month", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Year")
plt.xticks(ticks=np.arange(12) + 0.5, labels=month_labels, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# A6. REVENUE BY WEEKDAY
# ============================================================

daily_box = daily.copy()
daily_box["Revenue_scaled"] = daily_box["Revenue"] / REVENUE_SCALE
daily_box["weekday"] = pd.Categorical(
    daily_box["weekday"],
    categories=weekday_order,
    ordered=True,
)

plt.figure(figsize=(13, 6))

sns.boxplot(
    data=daily_box,
    x="weekday",
    y="Revenue_scaled",
    color="#9ecae1",
)

plt.title("Daily Revenue Distribution by Weekday", fontsize=16, fontweight="bold")
plt.xlabel("Weekday")
plt.ylabel(f"Daily Revenue ({REVENUE_UNIT})")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# A7. REVENUE DECOMPOSITION
# ============================================================

annual_diag = yearly[[
    "year", "Revenue", "Orders", "AOV", "Active_Customers", "ARPU",
    "YoY_Revenue_Growth", "YoY_Orders_Growth", "YoY_AOV_Growth",
    "YoY_Active_Customers_Growth", "YoY_ARPU_Growth"
]].copy()

def order_aov_driver(row):
    if pd.isna(row["YoY_Revenue_Growth"]):
        return "Base year"
    if row["YoY_Revenue_Growth"] >= 0:
        return "Revenue grew"
    if row["YoY_Orders_Growth"] < row["YoY_AOV_Growth"]:
        return "Main pressure: Orders decline"
    return "Main pressure: AOV decline"

def customer_arpu_driver(row):
    if pd.isna(row["YoY_Revenue_Growth"]):
        return "Base year"
    if row["YoY_Revenue_Growth"] >= 0:
        return "Revenue grew"
    if row["YoY_Active_Customers_Growth"] < row["YoY_ARPU_Growth"]:
        return "Main pressure: Active customers decline"
    return "Main pressure: ARPU decline"

annual_diag["Order_AOV_Driver"] = annual_diag.apply(order_aov_driver, axis=1)
annual_diag["Customer_ARPU_Driver"] = annual_diag.apply(customer_arpu_driver, axis=1)

display_table(
    annual_diag,
    money_cols=["Revenue", "AOV", "ARPU"],
    pct_cols=[
        "YoY_Revenue_Growth", "YoY_Orders_Growth", "YoY_AOV_Growth",
        "YoY_Active_Customers_Growth", "YoY_ARPU_Growth",
    ],
    int_cols=["Orders", "Active_Customers"],
)

decomp_long = annual_diag.melt(
    id_vars=["year"],
    value_vars=[
        "YoY_Revenue_Growth", "YoY_Orders_Growth", "YoY_AOV_Growth",
        "YoY_Active_Customers_Growth", "YoY_ARPU_Growth",
    ],
    var_name="Metric",
    value_name="YoY_Growth",
)

decomp_long["Metric"] = decomp_long["Metric"].replace({
    "YoY_Revenue_Growth": "Revenue",
    "YoY_Orders_Growth": "Orders",
    "YoY_AOV_Growth": "AOV",
    "YoY_Active_Customers_Growth": "Active_Customers",
    "YoY_ARPU_Growth": "ARPU",
})

plt.figure(figsize=(15, 6))

sns.lineplot(
    data=decomp_long,
    x="year",
    y="YoY_Growth",
    hue="Metric",
    marker="o",
    linewidth=2.3,
    palette=metric_palette,
)

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("YoY Growth Decomposition", fontsize=16, fontweight="bold")
plt.xlabel("Year")
plt.ylabel("YoY Growth")
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# A8. POST-2018 MOMENTUM
# ============================================================

if 2018 not in yearly["year"].values:
    raise ValueError("Không có năm 2018 trong dữ liệu.")

post_2018 = yearly[[
    "year", "Revenue", "Orders", "AOV", "Active_Customers", "ARPU",
    "Gross_Margin_Pct", "Discount_Rate"
]].copy()

base_2018 = post_2018.loc[post_2018["year"] == 2018].iloc[0]

for col in ["Revenue", "Orders", "AOV", "Active_Customers", "ARPU"]:
    post_2018[f"{col}_Index_2018_100"] = post_2018[col] / base_2018[col] * 100

post_2018_view = post_2018[post_2018["year"] >= 2016].copy()

display_table(
    post_2018_view,
    money_cols=["Revenue", "AOV", "ARPU"],
    pct_cols=["Gross_Margin_Pct", "Discount_Rate"],
    int_cols=["Orders", "Active_Customers"],
)

index_cols = [
    "Revenue_Index_2018_100",
    "Orders_Index_2018_100",
    "AOV_Index_2018_100",
    "Active_Customers_Index_2018_100",
    "ARPU_Index_2018_100",
]

post_long = post_2018_view.melt(
    id_vars=["year"],
    value_vars=index_cols,
    var_name="Metric",
    value_name="Index_2018_100",
)

post_long["Metric"] = post_long["Metric"].str.replace("_Index_2018_100", "", regex=False)

plt.figure(figsize=(14, 6))

sns.lineplot(
    data=post_long,
    x="year",
    y="Index_2018_100",
    hue="Metric",
    marker="o",
    linewidth=2.3,
    palette={k: metric_palette.get(k, "#333333") for k in post_long["Metric"].unique()},
)

plt.axhline(100, color="black", linestyle="--", linewidth=1)
plt.title("Post-2018 Momentum Diagnostic, 2018 = 100", fontsize=16, fontweight="bold")
plt.xlabel("Year")
plt.ylabel("Index, 2018 = 100")
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# A9. PLANNING CALENDAR
# ============================================================

planning_calendar = seasonality.copy()

planning_calendar["prep_month"] = planning_calendar["month"].apply(lambda m: 12 if m == 1 else m - 1)
planning_calendar["prep_month_name"] = planning_calendar["prep_month"].map(lambda m: month_labels[m - 1])

planning_calendar["Planning_Action"] = np.select(
    [
        planning_calendar["Seasonality_Class"].eq("Strong peak"),
        planning_calendar["Seasonality_Class"].eq("Mild peak"),
        planning_calendar["Seasonality_Class"].eq("Strong trough"),
        planning_calendar["Seasonality_Class"].eq("Mild trough"),
    ],
    [
        "Prepare inventory, campaign, warehouse capacity, and logistics one month earlier",
        "Prepare selective campaign and monitor inventory one month earlier",
        "Avoid overstock; use targeted demand stimulation if margin allows",
        "Keep lean inventory and check category-specific demand",
    ],
    default="Normal operating plan",
)

planning_calendar = planning_calendar[[
    "month", "month_name", "Seasonality_Index", "Seasonality_Class",
    "prep_month", "prep_month_name", "Avg_Revenue", "Avg_Orders",
    "Avg_AOV", "Avg_Active_Customers", "Avg_ARPU", "Planning_Action",
]].sort_values("Seasonality_Index", ascending=False)

display_table(
    planning_calendar,
    money_cols=["Avg_Revenue", "Avg_AOV", "Avg_ARPU"],
    pct_cols=["Seasonality_Index"],
    int_cols=["Avg_Orders", "Avg_Active_Customers"],
)

In [ ]:
# ============================================================
# A10. AUTO INSIGHTS - DEMAND & REVENUE SEASONALITY - TỰ RÚT INSIGHT CHO BÁO CÁO
# ============================================================

best_year_row = yearly.loc[yearly["Revenue"].idxmax()]
latest_year_row = yearly.sort_values("year").iloc[-1]
revenue_2018 = yearly.loc[yearly["year"] == 2018, "Revenue"].iloc[0]
latest_vs_2018 = latest_year_row["Revenue"] / revenue_2018 - 1

peak_months = planning_calendar[
    planning_calendar["Seasonality_Class"].isin(["Strong peak", "Mild peak"])
].copy()

trough_months = planning_calendar[
    planning_calendar["Seasonality_Class"].isin(["Strong trough", "Mild trough"])
].copy()

latest_driver = annual_diag.loc[
    annual_diag["year"] == latest_year_row["year"],
    ["Order_AOV_Driver", "Customer_ARPU_Driver"]
].iloc[0]

print("A. DEMAND & REVENUE SEASONALITY - KEY INSIGHTS")
print("-" * 70)
print(f"Peak revenue year: {int(best_year_row['year'])}, Revenue = {best_year_row['Revenue']:,.2f}")
print(f"Latest year: {int(latest_year_row['year'])}, Revenue = {latest_year_row['Revenue']:,.2f}")
print(f"Latest year revenue vs 2018: {latest_vs_2018:.2%}")
print()
print("Peak months based on internal seasonality:")
for _, r in peak_months.iterrows():
    print(f"- {r['month_name']}: Seasonality Index = {r['Seasonality_Index']:.2%}, prepare in {r['prep_month_name']}")
print()
print("Trough months based on internal seasonality:")
for _, r in trough_months.iterrows():
    print(f"- {r['month_name']}: Seasonality Index = {r['Seasonality_Index']:.2%}")
print()
print(f"Latest year diagnostic by Orders/AOV: {latest_driver['Order_AOV_Driver']}")
print(f"Latest year diagnostic by Active Customers/ARPU: {latest_driver['Customer_ARPU_Driver']}")

In [ ]:
# ============================================================
# B1. BUILD CATEGORY / PRODUCT / SEGMENT ECONOMICS
# ============================================================

category_summary = (
    fact
    .groupby("category", as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        COGS=("cogs_total", "sum"),
        Gross_Profit=("gross_profit", "sum"),
        Revenue_Before_Discount=("revenue_before_discount", "sum"),
        Discount=("discount_amount", "sum"),
        Units=("quantity", "sum"),
        Orders=("order_id", "nunique"),
        Customers=("customer_id", "nunique"),
        Products=("product_id", "nunique"),
    )
)

category_summary["Revenue_Share"] = category_summary["Revenue"] / category_summary["Revenue"].sum()
category_summary["Gross_Margin_Pct"] = category_summary["Gross_Profit"] / category_summary["Revenue"]
category_summary["Gross_Profit_Contribution_Share"] = category_summary["Gross_Profit"] / category_summary["Gross_Profit"].sum()
category_summary["Units_Share"] = category_summary["Units"] / category_summary["Units"].sum()
category_summary["Discount_Rate"] = category_summary["Discount"] / category_summary["Revenue_Before_Discount"]
category_summary["Revenue_per_Unit"] = category_summary["Revenue"] / category_summary["Units"]

category_summary = category_summary.sort_values("Revenue", ascending=False).reset_index(drop=True)
category_order = category_summary["category"].tolist()

category_summary["Concentration_Risk_Flag"] = np.select(
    [
        category_summary["Revenue_Share"] >= 0.70,
        category_summary["Revenue_Share"] >= 0.40,
        category_summary["Revenue_Share"] >= 0.20,
    ],
    [
        "Very high dependency",
        "High dependency",
        "Moderate dependency",
    ],
    default="Low dependency",
)

product_summary = (
    fact
    .groupby(["product_id", "product_name", "category", "segment"], as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        COGS=("cogs_total", "sum"),
        Gross_Profit=("gross_profit", "sum"),
        Units=("quantity", "sum"),
        Orders=("order_id", "nunique"),
        Customers=("customer_id", "nunique"),
    )
)

product_summary["Revenue_Share"] = product_summary["Revenue"] / product_summary["Revenue"].sum()
product_summary["Gross_Margin_Pct"] = product_summary["Gross_Profit"] / product_summary["Revenue"]
product_summary["Gross_Profit_Contribution_Share"] = product_summary["Gross_Profit"] / product_summary["Gross_Profit"].sum()
product_summary["Units_Share"] = product_summary["Units"] / product_summary["Units"].sum()

product_summary = product_summary.sort_values("Revenue", ascending=False).reset_index(drop=True)
product_summary["Rank"] = np.arange(1, len(product_summary) + 1)
product_summary["Cumulative_Revenue_Share"] = product_summary["Revenue_Share"].cumsum()

segment_summary = (
    fact
    .groupby(["category", "segment"], as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        COGS=("cogs_total", "sum"),
        Gross_Profit=("gross_profit", "sum"),
        Revenue_Before_Discount=("revenue_before_discount", "sum"),
        Discount=("discount_amount", "sum"),
        Units=("quantity", "sum"),
        Orders=("order_id", "nunique"),
        Customers=("customer_id", "nunique"),
        Products=("product_id", "nunique"),
    )
)

segment_summary["Revenue_Share"] = segment_summary["Revenue"] / segment_summary["Revenue"].sum()
segment_summary["Gross_Margin_Pct"] = segment_summary["Gross_Profit"] / segment_summary["Revenue"]
segment_summary["Gross_Profit_Contribution_Share"] = segment_summary["Gross_Profit"] / segment_summary["Gross_Profit"].sum()
segment_summary["Units_Share"] = segment_summary["Units"] / segment_summary["Units"].sum()
segment_summary["Discount_Rate"] = segment_summary["Discount"] / segment_summary["Revenue_Before_Discount"]
segment_summary["Revenue_per_Unit"] = segment_summary["Revenue"] / segment_summary["Units"]

segment_summary = segment_summary.sort_values("Revenue", ascending=False).reset_index(drop=True)

display_table(
    category_summary[[
        "category", "Revenue", "Revenue_Share",
        "COGS", "Gross_Profit", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share",
        "Units", "Units_Share", "Revenue_per_Unit",
        "Orders", "Customers", "Products",
        "Discount_Rate", "Concentration_Risk_Flag",
    ]],
    money_cols=["Revenue", "COGS", "Gross_Profit", "Revenue_per_Unit"],
    pct_cols=[
        "Revenue_Share", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share", "Units_Share", "Discount_Rate",
    ],
    int_cols=["Units", "Orders", "Customers", "Products"],
)

In [ ]:
# ============================================================
# B2. CATEGORY REVENUE SHARE + GROSS MARGIN - REVISED
# ============================================================

b2_df = category_summary.copy()
b2_df["category"] = b2_df["category"].astype(str)

category_order = b2_df.sort_values("Revenue_Share", ascending=False)["category"].tolist()
b2_palette = {cat: category_palette.get(cat, "#4C78A8") for cat in category_order}

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Revenue Share
sns.barplot(
    data=b2_df,
    x="category",
    y="Revenue_Share",
    order=category_order,
    hue="category",
    palette=b2_palette,
    dodge=False,
    legend=False,
    ax=axes[0],
)

axes[0].set_title("Revenue Share by Category", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Revenue Share")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

for patch in axes[0].patches:
    height = patch.get_height()
    if np.isfinite(height):
        axes[0].text(
            patch.get_x() + patch.get_width() / 2,
            height,
            f"{height:.1%}",
            ha="center",
            va="bottom",
            fontsize=10,
        )

# Gross Margin %
sns.barplot(
    data=b2_df,
    x="category",
    y="Gross_Margin_Pct",
    order=category_order,
    hue="category",
    palette=b2_palette,
    dodge=False,
    legend=False,
    ax=axes[1],
)

axes[1].set_title("Gross Margin % After Discount by Category", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Gross Margin %")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

for patch in axes[1].patches:
    height = patch.get_height()
    if np.isfinite(height):
        y_offset = 0.002 if height >= 0 else -0.002
        va = "bottom" if height >= 0 else "top"
        axes[1].text(
            patch.get_x() + patch.get_width() / 2,
            height + y_offset,
            f"{height:.1%}",
            ha="center",
            va=va,
            fontsize=10,
        )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# B3. CATEGORY PORTFOLIO BUBBLE CHART
# ============================================================

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=category_summary,
    x="Revenue_Share",
    y="Gross_Margin_Pct",
    size="Units",
    hue="category",
    palette=category_palette,
    sizes=(400, 3000),
    alpha=0.75,
    edgecolor="black",
    linewidth=0.8,
)

for _, row in category_summary.iterrows():
    plt.text(
        row["Revenue_Share"] + 0.005,
        row["Gross_Margin_Pct"] + 0.002,
        row["category"],
        fontsize=11,
        weight="bold",
    )

plt.axvline(category_summary["Revenue_Share"].median(), color="gray", linestyle="--", linewidth=1)
plt.axhline(category_summary["Gross_Margin_Pct"].median(), color="gray", linestyle="--", linewidth=1)

plt.title("Category Portfolio Matrix", fontsize=16, fontweight="bold")
plt.xlabel("Revenue Share")
plt.ylabel("Gross Margin % After Discount")
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.legend(title="Category / Units", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# B4. TOP PRODUCT CONCENTRATION
# ============================================================

def top_n_revenue_share(n):
    return product_summary.head(n)["Revenue_Share"].sum()

hhi_product = (product_summary["Revenue_Share"] ** 2).sum()

concentration_summary = pd.DataFrame({
    "metric": [
        "Top 1 product revenue share",
        "Top 5 product revenue share",
        "Top 10 product revenue share",
        "Top 20 product revenue share",
        "Top 50 product revenue share",
        "Product-level HHI",
    ],
    "value": [
        top_n_revenue_share(1),
        top_n_revenue_share(5),
        top_n_revenue_share(10),
        top_n_revenue_share(20),
        top_n_revenue_share(50),
        hhi_product,
    ],
})

top10_products = product_summary.head(10).copy()
top10_products["Product_Label"] = (
    top10_products["Rank"].astype(str)
    + ". "
    + top10_products["product_name"].astype(str)
)

display_table(concentration_summary, pct_cols=["value"])

display_table(
    top10_products[[
        "Rank", "product_id", "product_name", "category", "segment",
        "Revenue", "Revenue_Share", "Cumulative_Revenue_Share",
        "Gross_Profit", "Gross_Margin_Pct",
        "Units", "Orders", "Customers",
    ]],
    money_cols=["Revenue", "Gross_Profit"],
    pct_cols=["Revenue_Share", "Cumulative_Revenue_Share", "Gross_Margin_Pct"],
    int_cols=["Units", "Orders", "Customers"],
)

In [ ]:
# ============================================================
# B5. TOP 10 PRODUCTS BY REVENUE
# ============================================================

top10_plot = top10_products.sort_values("Revenue", ascending=True).copy()
top10_plot["Revenue_scaled"] = top10_plot["Revenue"] / REVENUE_SCALE

plt.figure(figsize=(12, 7))

sns.barplot(
    data=top10_plot,
    y="Product_Label",
    x="Revenue_scaled",
    hue="category",
    dodge=False,
    palette=category_palette,
)

plt.title("Top 10 Products by Revenue", fontsize=16, fontweight="bold")
plt.xlabel(f"Revenue ({REVENUE_UNIT})")
plt.ylabel("Product")
plt.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# B6. SEGMENT ECONOMICS
# ============================================================

display_table(
    segment_summary[[
        "category", "segment", "Revenue", "Revenue_Share",
        "Gross_Profit", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share",
        "Units", "Units_Share", "Revenue_per_Unit",
        "Orders", "Customers", "Products", "Discount_Rate",
    ]],
    money_cols=["Revenue", "Gross_Profit", "Revenue_per_Unit"],
    pct_cols=[
        "Revenue_Share", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share", "Units_Share", "Discount_Rate",
    ],
    int_cols=["Units", "Orders", "Customers", "Products"],
)

plt.figure(figsize=(13, 8))

sns.scatterplot(
    data=segment_summary,
    x="Revenue_Share",
    y="Gross_Margin_Pct",
    hue="category",
    size="Units",
    palette=category_palette,
    sizes=(120, 1800),
    alpha=0.75,
    edgecolor="black",
    linewidth=0.7,
)

for _, row in segment_summary.iterrows():
    plt.text(
        row["Revenue_Share"] + 0.001,
        row["Gross_Margin_Pct"] + 0.001,
        str(row["segment"]),
        fontsize=9,
    )

plt.axvline(segment_summary["Revenue_Share"].median(), color="gray", linestyle="--", linewidth=1)
plt.axhline(segment_summary["Gross_Margin_Pct"].median(), color="gray", linestyle="--", linewidth=1)

plt.title("Segment Portfolio Matrix", fontsize=16, fontweight="bold")
plt.xlabel("Revenue Share")
plt.ylabel("Gross Margin % After Discount")
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.legend(title="Category / Units", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# B7. SEGMENT REVENUE SHARE HEATMAP
# ============================================================

segment_heatmap = (
    segment_summary
    .pivot(index="category", columns="segment", values="Revenue_Share")
    .reindex(index=category_order)
)

plt.figure(figsize=(13, 5.5))

sns.heatmap(
    segment_heatmap,
    cmap="Blues",
    annot=True,
    fmt=".1%",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Revenue Share"},
)

plt.title("Revenue Share Heatmap by Category and Segment", fontsize=16, fontweight="bold")
plt.xlabel("Segment")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# B8. CATEGORY DEPENDENCY STRESS TEST
# ============================================================

leading_category = category_summary.iloc[0]["category"]
leading_revenue = category_summary.iloc[0]["Revenue"]
leading_gross_profit = category_summary.iloc[0]["Gross_Profit"]

total_revenue = category_summary["Revenue"].sum()
total_gross_profit = category_summary["Gross_Profit"].sum()

stress_test = []

for shock in [0.05, 0.10, 0.20, 0.30]:
    revenue_loss = leading_revenue * shock
    gross_profit_loss = leading_gross_profit * shock

    stress_test.append({
        "Leading_Category": leading_category,
        "Shock_to_Leading_Category": -shock,
        "Revenue_Loss": revenue_loss,
        "Total_Revenue_Change": -revenue_loss / total_revenue,
        "Gross_Profit_Loss": gross_profit_loss,
        "Total_Gross_Profit_Change": -gross_profit_loss / total_gross_profit,
    })

stress_test = pd.DataFrame(stress_test)

display_table(
    stress_test,
    money_cols=["Revenue_Loss", "Gross_Profit_Loss"],
    pct_cols=[
        "Shock_to_Leading_Category",
        "Total_Revenue_Change",
        "Total_Gross_Profit_Change",
    ],
)

stress_long = stress_test.melt(
    id_vars=["Shock_to_Leading_Category"],
    value_vars=["Total_Revenue_Change", "Total_Gross_Profit_Change"],
    var_name="Metric",
    value_name="Company_Impact",
)

stress_long["Metric"] = stress_long["Metric"].replace({
    "Total_Revenue_Change": "Total Revenue",
    "Total_Gross_Profit_Change": "Total Gross Profit",
})

stress_long["Shock_Label"] = stress_long["Shock_to_Leading_Category"].map(lambda x: f"{x:.0%}")

plt.figure(figsize=(10, 6))

sns.barplot(
    data=stress_long,
    x="Shock_Label",
    y="Company_Impact",
    hue="Metric",
    palette={
        "Total Revenue": "#1f77b4",
        "Total Gross Profit": "#d62728",
    },
)

plt.title(f"Stress Test: Impact if {leading_category} Weakens", fontsize=16, fontweight="bold")
plt.xlabel(f"Shock to {leading_category}")
plt.ylabel("Company-level Impact")
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.axhline(0, color="black", linewidth=1)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# B9. CATEGORY ACTION MATRIX
# ============================================================

rev_share_median = category_summary["Revenue_Share"].median()
margin_median = category_summary["Gross_Margin_Pct"].median()

action_matrix = category_summary.copy()

action_matrix["Portfolio_Position"] = np.select(
    [
        (action_matrix["Revenue_Share"] >= rev_share_median) &
        (action_matrix["Gross_Margin_Pct"] >= margin_median),

        (action_matrix["Revenue_Share"] >= rev_share_median) &
        (action_matrix["Gross_Margin_Pct"] < margin_median),

        (action_matrix["Revenue_Share"] < rev_share_median) &
        (action_matrix["Gross_Margin_Pct"] >= margin_median),

        (action_matrix["Revenue_Share"] < rev_share_median) &
        (action_matrix["Gross_Margin_Pct"] < margin_median),
    ],
    [
        "Scale / Protect",
        "Repair economics",
        "Test scaling",
        "Deprioritize / Reposition",
    ],
    default="Review",
)

action_matrix["Recommended_Action"] = np.select(
    [
        action_matrix["Portfolio_Position"].eq("Scale / Protect"),
        action_matrix["Portfolio_Position"].eq("Repair economics"),
        action_matrix["Portfolio_Position"].eq("Test scaling"),
        action_matrix["Portfolio_Position"].eq("Deprioritize / Reposition"),
    ],
    [
        "Protect availability, monitor stockout, avoid over-discounting, secure logistics capacity",
        "Audit COGS, discount depth, returns, and supplier terms before scaling further",
        "Run controlled campaigns and inventory tests to validate scalable demand",
        "Reduce broad promotion, reposition assortment, or keep as niche support",
    ],
    default="Manual review",
)

display_table(
    action_matrix[[
        "category", "Revenue", "Revenue_Share",
        "Gross_Profit", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share",
        "Units", "Units_Share",
        "Discount_Rate",
        "Concentration_Risk_Flag",
        "Portfolio_Position",
        "Recommended_Action",
    ]],
    money_cols=["Revenue", "Gross_Profit"],
    pct_cols=[
        "Revenue_Share", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share",
        "Units_Share", "Discount_Rate",
    ],
    int_cols=["Units"],
)

In [ ]:
# ============================================================
# B10. AUTO INSIGHTS - PRODUCT PORTFOLIO
# ============================================================

leading_cat = category_summary.iloc[0]
highest_margin_cat = category_summary.loc[category_summary["Gross_Margin_Pct"].idxmax()]
top10_share = product_summary.head(10)["Revenue_Share"].sum()
top20_share = product_summary.head(20)["Revenue_Share"].sum()

print("B. PRODUCT PORTFOLIO & CONCENTRATION RISK - KEY INSIGHTS")
print("-" * 70)
print(
    f"Leading category: {leading_cat['category']} "
    f"with {leading_cat['Revenue_Share']:.2%} revenue share "
    f"and {leading_cat['Gross_Profit_Contribution_Share']:.2%} gross profit contribution."
)
print(
    f"Highest gross margin category: {highest_margin_cat['category']} "
    f"with gross margin {highest_margin_cat['Gross_Margin_Pct']:.2%}."
)
print(f"Top 10 products account for {top10_share:.2%} of total revenue.")
print(f"Top 20 products account for {top20_share:.2%} of total revenue.")
print(f"Product-level HHI: {hhi_product:.4f}")
print()
print("Category action matrix:")
for _, r in action_matrix.iterrows():
    print(f"- {r['category']}: {r['Portfolio_Position']} | {r['Recommended_Action']}")

In [ ]:
# ============================================================
# B11. CATEGORY ACTION MATRIX
# ============================================================

rev_share_median = category_summary["Revenue_Share"].median()
margin_median = category_summary["Gross_Margin_Pct"].median()

action_matrix = category_summary.copy()

action_matrix["Portfolio_Position"] = np.select(
    [
        (action_matrix["Revenue_Share"] >= rev_share_median) & (action_matrix["Gross_Margin_Pct"] >= margin_median),
        (action_matrix["Revenue_Share"] >= rev_share_median) & (action_matrix["Gross_Margin_Pct"] < margin_median),
        (action_matrix["Revenue_Share"] < rev_share_median) & (action_matrix["Gross_Margin_Pct"] >= margin_median),
        (action_matrix["Revenue_Share"] < rev_share_median) & (action_matrix["Gross_Margin_Pct"] < margin_median)
    ],
    [
        "Scale / Protect",
        "Repair economics",
        "Test scaling",
        "Deprioritize / Reposition"
    ],
    default="Review"
)

action_matrix["Recommended_Action"] = np.select(
    [
        action_matrix["Portfolio_Position"].eq("Scale / Protect"),
        action_matrix["Portfolio_Position"].eq("Repair economics"),
        action_matrix["Portfolio_Position"].eq("Test scaling"),
        action_matrix["Portfolio_Position"].eq("Deprioritize / Reposition")
    ],
    [
        "Protect availability, monitor stockout, avoid over-discounting, secure logistics capacity",
        "Audit COGS, discount depth, returns, and supplier terms before scaling further",
        "Run controlled campaigns and inventory tests to validate scalable demand",
        "Reduce broad promotion, reposition assortment, or keep as niche support"
    ],
    default="Manual review"
)

display_table(
    action_matrix[[
        "category", "Revenue", "Revenue_Share",
        "Gross_Profit", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share",
        "Units", "Units_Share",
        "Discount_Rate",
        "Concentration_Risk_Flag",
        "Portfolio_Position",
        "Recommended_Action"
    ]],
    money_cols=["Revenue", "Gross_Profit"],
    pct_cols=[
        "Revenue_Share", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share",
        "Units_Share", "Discount_Rate"
    ],
    int_cols=["Units"]
)

In [ ]:
# ============================================================
# FINAL REPORT TABLES
# ============================================================

A_seasonality_report = planning_calendar[[
    "month", "month_name", "Seasonality_Index", "Seasonality_Class",
    "prep_month_name", "Avg_Revenue", "Avg_Orders", "Avg_AOV",
    "Avg_Active_Customers", "Avg_ARPU", "Planning_Action",
]].copy()

A_post2018_report = post_2018_view[[
    "year",
    "Revenue_Index_2018_100",
    "Orders_Index_2018_100",
    "AOV_Index_2018_100",
    "Active_Customers_Index_2018_100",
    "ARPU_Index_2018_100",
    "Gross_Margin_Pct",
    "Discount_Rate",
]].copy()

B_category_report = category_summary[[
    "category", "Revenue_Share", "Gross_Margin_Pct",
    "Gross_Profit_Contribution_Share", "Units_Share",
    "Revenue", "Gross_Profit", "Units",
    "Concentration_Risk_Flag",
]].copy()

B_action_report = action_matrix[[
    "category", "Portfolio_Position", "Recommended_Action",
    "Revenue_Share", "Gross_Margin_Pct",
    "Gross_Profit_Contribution_Share",
    "Discount_Rate",
]].copy()

print("A. SEASONALITY PLANNING TABLE")
display_table(
    A_seasonality_report,
    money_cols=["Avg_Revenue", "Avg_AOV", "Avg_ARPU"],
    pct_cols=["Seasonality_Index"],
    int_cols=["Avg_Orders", "Avg_Active_Customers"],
)

print("A. POST-2018 DIAGNOSTIC TABLE")
display_table(
    A_post2018_report,
    pct_cols=["Gross_Margin_Pct", "Discount_Rate"],
)

print("B. CATEGORY ECONOMICS TABLE")
display_table(
    B_category_report,
    money_cols=["Revenue", "Gross_Profit"],
    pct_cols=[
        "Revenue_Share", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share", "Units_Share",
    ],
    int_cols=["Units"],
)

print("B. PRODUCT CONCENTRATION TABLE")
display_table(
    concentration_summary,
    pct_cols=["value"],
)

print("B. CATEGORY ACTION MATRIX")
display_table(
    B_action_report,
    pct_cols=[
        "Revenue_Share", "Gross_Margin_Pct",
        "Gross_Profit_Contribution_Share", "Discount_Rate",
    ],
)

In [ ]:
# ============================================================
# C0. PREPARE PROFIT / RETURNS DATASETS
# ============================================================

returns_enriched = (
    returns
    .merge(
        products[["product_id", "product_name", "category", "segment", "size", "color"]],
        on="product_id",
        how="left",
    )
    .merge(
        orders[["order_id", "order_date", "customer_id", "order_source", "device_type"]],
        on="order_id",
        how="left",
    )
)

for c in ["refund_amount", "return_quantity"]:
    returns_enriched[c] = pd.to_numeric(returns_enriched[c], errors="coerce").fillna(0)

returns_enriched["return_reason"] = returns_enriched["return_reason"].fillna("unknown")
returns_enriched["return_month"] = returns_enriched["return_date"].dt.to_period("M").dt.to_timestamp()

sales_monthly = (
    sales
    .groupby(pd.Grouper(key="Date", freq="MS"))
    .agg(
        Reported_Revenue=("Revenue", "sum"),
        Reported_COGS=("COGS", "sum"),
    )
    .reset_index()
    .rename(columns={"Date": "date"})
)

sales_monthly["Reported_Gross_Profit"] = (
    sales_monthly["Reported_Revenue"] - sales_monthly["Reported_COGS"]
)
sales_monthly["Reported_GM_Pct"] = safe_divide(
    sales_monthly["Reported_Gross_Profit"],
    sales_monthly["Reported_Revenue"],
)

monthly_refunds = (
    returns_enriched
    .groupby("return_month", as_index=False)
    .agg(
        Refund_Amount=("refund_amount", "sum"),
        Return_Units=("return_quantity", "sum"),
        Return_Orders=("order_id", "nunique"),
    )
    .rename(columns={"return_month": "date"})
)

monthly_profit = (
    monthly[[
        "date", "period", "year", "month", "month_name",
        "Revenue", "COGS", "Gross_Profit",
        "Revenue_Before_Discount", "Discount",
        "Orders", "Active_Customers", "AOV", "ARPU",
        "Gross_Margin_Pct", "Discount_Rate",
    ]]
    .merge(sales_monthly, on="date", how="left")
    .merge(monthly_refunds, on="date", how="left")
)

for c in ["Refund_Amount", "Return_Units", "Return_Orders"]:
    monthly_profit[c] = monthly_profit[c].fillna(0)

monthly_profit["Commercial_Revenue"] = monthly_profit["Revenue"]
monthly_profit["Commercial_GM_Pct"] = safe_divide(
    monthly_profit["Gross_Profit"],
    monthly_profit["Commercial_Revenue"],
)
monthly_profit["COGS_Rate"] = safe_divide(
    monthly_profit["COGS"],
    monthly_profit["Commercial_Revenue"],
)
monthly_profit["Refund_to_Revenue"] = safe_divide(
    monthly_profit["Refund_Amount"],
    monthly_profit["Commercial_Revenue"],
)
monthly_profit["Refund_to_Gross_Profit"] = safe_divide(
    monthly_profit["Refund_Amount"],
    monthly_profit["Gross_Profit"],
)
monthly_profit["Margin_After_Returns"] = (
    monthly_profit["Gross_Profit"] - monthly_profit["Refund_Amount"]
)
monthly_profit["Margin_After_Returns_Pct"] = safe_divide(
    monthly_profit["Margin_After_Returns"],
    monthly_profit["Commercial_Revenue"],
)
monthly_profit["Negative_Commercial_GM_Flag"] = monthly_profit["Commercial_GM_Pct"] < 0
monthly_profit["Negative_Margin_After_Returns_Flag"] = (
    monthly_profit["Margin_After_Returns_Pct"] < 0
)

returns_by_reason = (
    returns_enriched
    .groupby("return_reason", as_index=False)
    .agg(
        Refund_Amount=("refund_amount", "sum"),
        Return_Units=("return_quantity", "sum"),
        Return_Orders=("order_id", "nunique"),
    )
    .sort_values("Refund_Amount", ascending=False)
    .reset_index(drop=True)
)

returns_by_reason["Refund_Share"] = safe_divide(
    returns_by_reason["Refund_Amount"],
    returns_by_reason["Refund_Amount"].sum(),
)

category_refunds = (
    returns_enriched
    .groupby("category", as_index=False)
    .agg(
        Refund_Amount=("refund_amount", "sum"),
        Return_Units=("return_quantity", "sum"),
        Return_Orders=("order_id", "nunique"),
    )
)

category_profit = (
    fact
    .groupby("category", as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        Revenue_Before_Discount=("revenue_before_discount", "sum"),
        Discount=("discount_amount", "sum"),
        COGS=("cogs_total", "sum"),
        Gross_Profit=("gross_profit", "sum"),
        Units=("quantity", "sum"),
        Orders=("order_id", "nunique"),
        Customers=("customer_id", "nunique"),
    )
    .merge(category_refunds, on="category", how="left")
)

for c in ["Refund_Amount", "Return_Units", "Return_Orders"]:
    category_profit[c] = category_profit[c].fillna(0)

category_profit["Gross_Margin_Pct"] = safe_divide(
    category_profit["Gross_Profit"],
    category_profit["Revenue"],
)
category_profit["Discount_Rate"] = safe_divide(
    category_profit["Discount"],
    category_profit["Revenue_Before_Discount"],
)
category_profit["Refund_to_Revenue"] = safe_divide(
    category_profit["Refund_Amount"],
    category_profit["Revenue"],
)
category_profit["Refund_to_Gross_Profit"] = safe_divide(
    category_profit["Refund_Amount"],
    category_profit["Gross_Profit"],
)
category_profit["Margin_After_Returns"] = (
    category_profit["Gross_Profit"] - category_profit["Refund_Amount"]
)
category_profit["Margin_After_Returns_Pct"] = safe_divide(
    category_profit["Margin_After_Returns"],
    category_profit["Revenue"],
)
category_profit["Revenue_Share"] = safe_divide(
    category_profit["Revenue"],
    category_profit["Revenue"].sum(),
)
category_profit = category_profit.sort_values("Revenue", ascending=False).reset_index(drop=True)

category_month_profit = (
    fact
    .groupby([pd.Grouper(key="date", freq="MS"), "category"], as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        Revenue_Before_Discount=("revenue_before_discount", "sum"),
        Discount=("discount_amount", "sum"),
        COGS=("cogs_total", "sum"),
        Gross_Profit=("gross_profit", "sum"),
        Units=("quantity", "sum"),
        Orders=("order_id", "nunique"),
    )
)

category_month_refunds = (
    returns_enriched
    .groupby(["return_month", "category"], as_index=False)
    .agg(
        Refund_Amount=("refund_amount", "sum"),
        Return_Units=("return_quantity", "sum"),
    )
    .rename(columns={"return_month": "date"})
)

category_month_profit = category_month_profit.merge(
    category_month_refunds,
    on=["date", "category"],
    how="left",
)

for c in ["Refund_Amount", "Return_Units"]:
    category_month_profit[c] = category_month_profit[c].fillna(0)

category_month_profit["Gross_Margin_Pct"] = safe_divide(
    category_month_profit["Gross_Profit"],
    category_month_profit["Revenue"],
)
category_month_profit["Discount_Rate"] = safe_divide(
    category_month_profit["Discount"],
    category_month_profit["Revenue_Before_Discount"],
)
category_month_profit["Margin_After_Returns"] = (
    category_month_profit["Gross_Profit"] - category_month_profit["Refund_Amount"]
)
category_month_profit["Margin_After_Returns_Pct"] = safe_divide(
    category_month_profit["Margin_After_Returns"],
    category_month_profit["Revenue"],
)
category_month_profit["period"] = category_month_profit["date"].dt.to_period("M").astype(str)
category_month_profit["Revenue_Pctl"] = category_month_profit["Revenue"].rank(pct=True)

high_revenue_bad_profit = (
    category_month_profit[
        (category_month_profit["Revenue_Pctl"] >= 0.75) &
        (category_month_profit["Margin_After_Returns_Pct"] < 0)
    ]
    .sort_values(["Revenue", "Margin_After_Returns_Pct"], ascending=[False, True])
    .head(20)
)

display(monthly_profit.head())
display(category_profit.head())

In [ ]:
# ============================================================
# C1. SUMMARY TABLES
# ============================================================

negative_commercial_gm_months = int(monthly_profit["Negative_Commercial_GM_Flag"].sum())
negative_margin_after_returns_months = int(
    monthly_profit["Negative_Margin_After_Returns_Flag"].sum()
)

profit_overview = pd.DataFrame({
    "Metric": [
        "Total commercial revenue",
        "Total discount",
        "Total COGS",
        "Total gross profit",
        "Total refund amount",
        "Commercial GM %",
        "Margin after returns %",
        "Negative commercial GM months",
        "Negative margin-after-returns months",
    ],
    "Value": [
        monthly_profit["Commercial_Revenue"].sum(),
        monthly_profit["Discount"].sum(),
        monthly_profit["COGS"].sum(),
        monthly_profit["Gross_Profit"].sum(),
        monthly_profit["Refund_Amount"].sum(),
        monthly_profit["Gross_Profit"].sum() / monthly_profit["Commercial_Revenue"].sum(),
        (monthly_profit["Gross_Profit"].sum() - monthly_profit["Refund_Amount"].sum()) / monthly_profit["Commercial_Revenue"].sum(),
        negative_commercial_gm_months,
        negative_margin_after_returns_months,
    ]
})

print("PROFIT OVERVIEW")
display(profit_overview)

print("MONTHLY PROFIT TABLE - LAST 24 MONTHS")
display_table(
    monthly_profit[[
        "period", "Commercial_Revenue", "Discount", "COGS", "Gross_Profit",
        "Commercial_GM_Pct", "Refund_Amount", "Refund_to_Revenue",
        "Refund_to_Gross_Profit", "Margin_After_Returns",
        "Margin_After_Returns_Pct", "Discount_Rate", "COGS_Rate",
        "Orders", "AOV",
    ]].tail(24),
    money_cols=[
        "Commercial_Revenue", "Discount", "COGS", "Gross_Profit",
        "Refund_Amount", "Margin_After_Returns", "AOV"
    ],
    pct_cols=[
        "Commercial_GM_Pct", "Refund_to_Revenue",
        "Refund_to_Gross_Profit", "Margin_After_Returns_Pct",
        "Discount_Rate", "COGS_Rate"
    ],
    int_cols=["Orders"],
)

print("CATEGORY PROFIT TABLE")
display_table(
    category_profit[[
        "category", "Revenue", "Revenue_Share",
        "Discount", "Discount_Rate", "COGS", "Gross_Profit",
        "Gross_Margin_Pct", "Refund_Amount", "Refund_to_Revenue",
        "Refund_to_Gross_Profit", "Margin_After_Returns",
        "Margin_After_Returns_Pct", "Orders", "Units"
    ]],
    money_cols=[
        "Revenue", "Discount", "COGS", "Gross_Profit",
        "Refund_Amount", "Margin_After_Returns"
    ],
    pct_cols=[
        "Revenue_Share", "Discount_Rate", "Gross_Margin_Pct",
        "Refund_to_Revenue", "Refund_to_Gross_Profit",
        "Margin_After_Returns_Pct"
    ],
    int_cols=["Orders", "Units"],
)

print("HIGH-REVENUE CATEGORY-MONTHS WITH NEGATIVE MARGIN AFTER RETURNS")
display_table(
    high_revenue_bad_profit[[
        "period", "category", "Revenue", "Discount", "COGS", "Gross_Profit",
        "Refund_Amount", "Margin_After_Returns", "Margin_After_Returns_Pct",
        "Discount_Rate", "Orders", "Units"
    ]],
    money_cols=[
        "Revenue", "Discount", "COGS", "Gross_Profit",
        "Refund_Amount", "Margin_After_Returns"
    ],
    pct_cols=["Margin_After_Returns_Pct", "Discount_Rate"],
    int_cols=["Orders", "Units"],
)

In [ ]:
# ============================================================
# C2. WATERFALL CHART
# ============================================================

waterfall_values = {
    "Reported Revenue": fact["revenue_before_discount"].sum(),
    "Discount": -fact["discount_amount"].sum(),
    "Commercial Revenue": fact["item_revenue"].sum(),
    "COGS": -fact["cogs_total"].sum(),
    "Gross Margin": fact["gross_profit"].sum(),
    "Refund": -returns_enriched["refund_amount"].sum(),
    "Margin after Returns": fact["gross_profit"].sum() - returns_enriched["refund_amount"].sum(),
}

waterfall_steps = [
    ("Reported Revenue", waterfall_values["Reported Revenue"], "absolute"),
    ("Discount", waterfall_values["Discount"], "relative"),
    ("Commercial Revenue", waterfall_values["Commercial Revenue"], "subtotal"),
    ("COGS", waterfall_values["COGS"], "relative"),
    ("Gross Margin", waterfall_values["Gross Margin"], "subtotal"),
    ("Refund", waterfall_values["Refund"], "relative"),
    ("Margin after Returns", waterfall_values["Margin after Returns"], "total"),
]

labels = []
bottoms = []
heights = []
colors = []
running = 0

for label, value, step_type in waterfall_steps:
    labels.append(label)

    if step_type == "absolute":
        bottoms.append(0)
        heights.append(value)
        running = value
        colors.append("#4C78A8")

    elif step_type == "relative":
        if value >= 0:
            bottoms.append(running)
            heights.append(value)
            running += value
            colors.append("#54A24B")
        else:
            bottoms.append(running + value)
            heights.append(-value)
            running += value
            colors.append("#E45756")

    elif step_type in ["subtotal", "total"]:
        bottoms.append(0)
        heights.append(value)
        running = value
        colors.append("#72B7B2")

plt.figure(figsize=(14, 6))
plt.bar(labels, heights, bottom=bottoms, color=colors, edgecolor="black")

for i, (b, h, lab) in enumerate(zip(bottoms, heights, labels)):
    plt.text(
        i,
        b + h + (0.02 * max(heights)),
        f"{h / REVENUE_SCALE:,.1f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

plt.title("Profit Leakage Waterfall", fontsize=16, fontweight="bold")
plt.ylabel(f"Amount ({REVENUE_UNIT})")
plt.xticks(rotation=20)
plt.gca().yaxis.set_major_formatter(mtick.FuncFormatter(money_axis))
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# C3. MONTHLY GM% TREND
# ============================================================

gm_plot = monthly_profit.copy()
gm_long = gm_plot.melt(
    id_vars=["date"],
    value_vars=["Reported_GM_Pct", "Commercial_GM_Pct", "Margin_After_Returns_Pct"],
    var_name="Metric",
    value_name="GM_Pct",
)

gm_long["Metric"] = gm_long["Metric"].replace({
    "Reported_GM_Pct": "Reported GM %",
    "Commercial_GM_Pct": "Commercial GM %",
    "Margin_After_Returns_Pct": "Margin after Returns %",
})

plt.figure(figsize=(15, 6))
sns.lineplot(
    data=gm_long,
    x="date",
    y="GM_Pct",
    hue="Metric",
    linewidth=2.2,
    palette={
        "Reported GM %": "#1f77b4",
        "Commercial GM %": "#ff7f0e",
        "Margin after Returns %": "#d62728",
    },
)

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Monthly Gross Margin Trend", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Margin %")
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# C4. SEASONAL GM% BY CALENDAR MONTH
# ============================================================

gm_seasonality = (
    monthly_profit
    .groupby(["month", "month_name"], as_index=False)
    .agg(
        Avg_Commercial_GM_Pct=("Commercial_GM_Pct", "mean"),
        Avg_Margin_After_Returns_Pct=("Margin_After_Returns_Pct", "mean"),
        Avg_Discount_Rate=("Discount_Rate", "mean"),
        Avg_Refund_to_Revenue=("Refund_to_Revenue", "mean"),
        Avg_Revenue=("Commercial_Revenue", "mean"),
    )
    .sort_values("month")
)

display_table(
    gm_seasonality,
    money_cols=["Avg_Revenue"],
    pct_cols=[
        "Avg_Commercial_GM_Pct",
        "Avg_Margin_After_Returns_Pct",
        "Avg_Discount_Rate",
        "Avg_Refund_to_Revenue",
    ],
)

plt.figure(figsize=(13, 6))
sns.barplot(
    data=gm_seasonality,
    x="month_name",
    y="Avg_Margin_After_Returns_Pct",
    color="#4C78A8",
)

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Average Margin after Returns % by Calendar Month", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Average Margin after Returns %")
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# C5. REFUND BY RETURN REASON
# ============================================================

top_return_reasons = returns_by_reason.head(10).copy()
top_return_reasons["Refund_Amount_scaled"] = top_return_reasons["Refund_Amount"] / REVENUE_SCALE

display_table(
    returns_by_reason,
    money_cols=["Refund_Amount"],
    pct_cols=["Refund_Share"],
    int_cols=["Return_Units", "Return_Orders"],
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_return_reasons.sort_values("Refund_Amount_scaled", ascending=True),
    x="Refund_Amount_scaled",
    y="return_reason",
    color="#E45756",
)

plt.title("Top Return Reasons by Refund Amount", fontsize=16, fontweight="bold")
plt.xlabel(f"Refund Amount ({REVENUE_UNIT})")
plt.ylabel("Return Reason")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# C6. AVERAGE CALENDAR-MONTH DISCOUNT RATE VS GM%
# ============================================================

month_avg = (
    monthly_profit
    .groupby("month", as_index=False)
    .agg(
        Avg_Discount_Rate=("Discount_Rate", "mean"),
        Avg_Margin_After_Returns_Pct=("Margin_After_Returns_Pct", "mean"),
        Avg_Commercial_GM_Pct=("Commercial_GM_Pct", "mean"),
        Avg_Revenue=("Commercial_Revenue", "mean"),
        Month_Count=("period", "nunique"),
    )
)

month_avg["month_name"] = month_avg["month"].map(lambda m: month_labels[m - 1])
month_avg["Avg_Revenue_scaled"] = month_avg["Avg_Revenue"] / REVENUE_SCALE

display_table(
    month_avg[[
        "month", "month_name", "Avg_Discount_Rate",
        "Avg_Commercial_GM_Pct", "Avg_Margin_After_Returns_Pct",
        "Avg_Revenue", "Month_Count"
    ]],
    money_cols=["Avg_Revenue"],
    pct_cols=[
        "Avg_Discount_Rate",
        "Avg_Commercial_GM_Pct",
        "Avg_Margin_After_Returns_Pct"
    ],
    int_cols=["Month_Count"],
)

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=month_avg,
    x="Avg_Discount_Rate",
    y="Avg_Margin_After_Returns_Pct",
    size="Avg_Revenue_scaled",
    sizes=(120, 900),
    color="#9467bd",
    alpha=0.78,
    edgecolor="black",
    linewidth=0.8,
)

for _, row in month_avg.iterrows():
    plt.text(
        row["Avg_Discount_Rate"] + 0.001,
        row["Avg_Margin_After_Returns_Pct"] + 0.001,
        row["month_name"],
        fontsize=10,
        weight="bold",
    )

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Average Calendar-Month Discount Rate vs Margin after Returns %", fontsize=16, fontweight="bold")
plt.xlabel("Average Discount Rate by Calendar Month")
plt.ylabel("Average Margin after Returns % by Calendar Month")
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# C7. GUARDRAILS + AUTO INSIGHTS
# ============================================================

positive_months = monthly_profit[monthly_profit["Margin_After_Returns_Pct"] > 0].copy()

guardrails = pd.DataFrame({
    "Guardrail": [
        "Monthly discount rate ceiling",
        "Monthly refund / revenue warning line",
        "Monthly COGS / revenue warning line",
        "Monthly margin-after-returns floor",
    ],
    "Suggested_Threshold": [
        positive_months["Discount_Rate"].quantile(0.75),
        positive_months["Refund_to_Revenue"].quantile(0.75),
        positive_months["COGS_Rate"].quantile(0.75),
        positive_months["Margin_After_Returns_Pct"].quantile(0.25),
    ],
    "Interpretation": [
        "Above this level, risk of buying revenue with discount increases",
        "Above this level, refund leakage becomes unusually large",
        "Above this level, product mix / sourcing quality may be deteriorating",
        "If projected margin is below this floor, trigger profit-protection actions",
    ],
})

mix_guardrail = category_profit.copy()
mix_guardrail["Mix_Risk_Flag"] = np.select(
    [
        (mix_guardrail["Revenue_Share"] >= mix_guardrail["Revenue_Share"].median()) &
        (mix_guardrail["Margin_After_Returns_Pct"] < mix_guardrail["Margin_After_Returns_Pct"].median()),
        (mix_guardrail["Refund_to_Revenue"] >= mix_guardrail["Refund_to_Revenue"].median()),
    ],
    [
        "High revenue but low-quality profit",
        "High return leakage",
    ],
    default="Normal",
)

display_table(guardrails, pct_cols=["Suggested_Threshold"])

print("PRODUCT MIX GUARDRAIL")
display_table(
    mix_guardrail[[
        "category", "Revenue_Share", "Gross_Margin_Pct",
        "Refund_to_Revenue", "Margin_After_Returns_Pct",
        "Mix_Risk_Flag"
    ]],
    pct_cols=[
        "Revenue_Share", "Gross_Margin_Pct",
        "Refund_to_Revenue", "Margin_After_Returns_Pct"
    ],
)

top3_return_share = returns_by_reason.head(3)["Refund_Share"].sum()
worst_margin_month = monthly_profit.loc[monthly_profit["Margin_After_Returns_Pct"].idxmin()]

print("C. PROFIT QUALITY & LEAKAGE - KEY INSIGHTS")
print("-" * 70)
print(f"Commercial GM % (overall): {monthly_profit['Gross_Profit'].sum() / monthly_profit['Commercial_Revenue'].sum():.2%}")
print(f"Margin after Returns % (overall): {(monthly_profit['Gross_Profit'].sum() - monthly_profit['Refund_Amount'].sum()) / monthly_profit['Commercial_Revenue'].sum():.2%}")
print(f"Negative commercial GM months: {negative_commercial_gm_months}")
print(f"Negative margin-after-returns months: {negative_margin_after_returns_months}")
print(f"Refund as % of gross profit: {returns_enriched['refund_amount'].sum() / monthly_profit['Gross_Profit'].sum():.2%}")
print(f"Top 3 return reasons share of refund: {top3_return_share:.2%}")
print(
    f"Worst month after returns: {worst_margin_month['period']} | "
    f"Margin after Returns % = {worst_margin_month['Margin_After_Returns_Pct']:.2%} | "
    f"Discount Rate = {worst_margin_month['Discount_Rate']:.2%}"
)

In [ ]:
# ============================================================
# D0. PREPARE PROMO / TRAFFIC DATASETS
# ============================================================

promo_palette = {
    "Promo": "#d62728",
    "Non-Promo": "#1f77b4",
}

fact["promo_flag"] = (
    fact["promo_id"].notna() |
    fact["promo_id_2"].notna() |
    (fact["discount_amount"].fillna(0) > 0)
)
fact["promo_group"] = np.where(fact["promo_flag"], "Promo", "Non-Promo")

promo_summary = (
    fact
    .groupby("promo_group", as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        Revenue_Before_Discount=("revenue_before_discount", "sum"),
        Discount=("discount_amount", "sum"),
        COGS=("cogs_total", "sum"),
        Gross_Profit=("gross_profit", "sum"),
        Orders=("order_id", "nunique"),
        Customers=("customer_id", "nunique"),
        Units=("quantity", "sum"),
    )
)

promo_summary["Revenue_Share"] = safe_divide(
    promo_summary["Revenue"],
    promo_summary["Revenue"].sum(),
)
promo_summary["Order_Share"] = safe_divide(
    promo_summary["Orders"],
    promo_summary["Orders"].sum(),
)
promo_summary["Gross_Margin_Pct"] = safe_divide(
    promo_summary["Gross_Profit"],
    promo_summary["Revenue"],
)
promo_summary["Discount_Rate"] = safe_divide(
    promo_summary["Discount"],
    promo_summary["Revenue_Before_Discount"],
)
promo_summary["AOV"] = safe_divide(promo_summary["Revenue"], promo_summary["Orders"])
promo_summary["ARPU"] = safe_divide(promo_summary["Revenue"], promo_summary["Customers"])

monthly_promo_eff = (
    fact
    .groupby([pd.Grouper(key="date", freq="MS"), "promo_group"], as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        Revenue_Before_Discount=("revenue_before_discount", "sum"),
        Discount=("discount_amount", "sum"),
        COGS=("cogs_total", "sum"),
        Gross_Profit=("gross_profit", "sum"),
        Orders=("order_id", "nunique"),
        Customers=("customer_id", "nunique"),
    )
)

monthly_promo_eff["Gross_Margin_Pct"] = safe_divide(
    monthly_promo_eff["Gross_Profit"],
    monthly_promo_eff["Revenue"],
)
monthly_promo_eff["Discount_Rate"] = safe_divide(
    monthly_promo_eff["Discount"],
    monthly_promo_eff["Revenue_Before_Discount"],
)
monthly_promo_eff["AOV"] = safe_divide(
    monthly_promo_eff["Revenue"],
    monthly_promo_eff["Orders"],
)
monthly_promo_eff["ARPU"] = safe_divide(
    monthly_promo_eff["Revenue"],
    monthly_promo_eff["Customers"],
)
monthly_promo_eff["period"] = monthly_promo_eff["date"].dt.to_period("M").astype(str)

promo_fact = fact.copy()
promo_fact["promo_list"] = promo_fact[["promo_id", "promo_id_2"]].apply(
    lambda row: [x for x in pd.unique(row.dropna()) if pd.notna(x)],
    axis=1,
)
promo_fact["promo_count"] = promo_fact["promo_list"].apply(lambda x: len(x))
promo_fact = promo_fact[promo_fact["promo_count"] > 0].copy()

promo_fact_exploded = promo_fact.explode("promo_list").rename(columns={"promo_list": "promo_id_attributed"})

for c in [
    "item_revenue", "revenue_before_discount", "discount_amount",
    "cogs_total", "gross_profit", "quantity"
]:
    promo_fact_exploded[f"alloc_{c}"] = promo_fact_exploded[c] / promo_fact_exploded["promo_count"]

promotions_ref = promotions.copy().rename(columns={
    "promo_id": "promo_id_attributed",
    "promo_name": "promo_name_ref",
    "promo_type": "promo_type_ref",
    "discount_value": "discount_value_ref",
    "min_order_value": "min_order_value_ref",
    "applicable_category": "applicable_category_ref",
    "promo_channel": "promo_channel_ref",
    "stackable_flag": "stackable_flag_ref",
})

promo_fact_exploded = promo_fact_exploded.merge(
    promotions_ref,
    on="promo_id_attributed",
    how="left",
)

promo_fact_exploded["promo_label"] = promo_fact_exploded["promo_name_ref"].fillna(
    promo_fact_exploded["promo_id_attributed"].astype(str)
)

promo_campaign_summary = (
    promo_fact_exploded
    .groupby(
        [
            "promo_id_attributed", "promo_label", "promo_type_ref",
            "min_order_value_ref", "applicable_category_ref"
        ],
        as_index=False
    )
    .agg(
        Revenue=("alloc_item_revenue", "sum"),
        Revenue_Before_Discount=("alloc_revenue_before_discount", "sum"),
        Discount=("alloc_discount_amount", "sum"),
        COGS=("alloc_cogs_total", "sum"),
        Gross_Profit=("alloc_gross_profit", "sum"),
        Units=("alloc_quantity", "sum"),
        Orders=("order_id", "nunique"),
        Customers=("customer_id", "nunique"),
    )
)

promo_campaign_summary["Revenue_Share"] = safe_divide(
    promo_campaign_summary["Revenue"],
    promo_campaign_summary["Revenue"].sum(),
)
promo_campaign_summary["Gross_Margin_Pct"] = safe_divide(
    promo_campaign_summary["Gross_Profit"],
    promo_campaign_summary["Revenue"],
)
promo_campaign_summary["Discount_Rate"] = safe_divide(
    promo_campaign_summary["Discount"],
    promo_campaign_summary["Revenue_Before_Discount"],
)
promo_campaign_summary["AOV"] = safe_divide(
    promo_campaign_summary["Revenue"],
    promo_campaign_summary["Orders"],
)
promo_campaign_summary["ARPU"] = safe_divide(
    promo_campaign_summary["Revenue"],
    promo_campaign_summary["Customers"],
)

nonpromo_gm = promo_summary.loc[promo_summary["promo_group"] == "Non-Promo", "Gross_Margin_Pct"].iloc[0]
promo_revenue_median = promo_campaign_summary["Revenue"].median()

promo_campaign_summary["Recommendation"] = np.select(
    [
        (promo_campaign_summary["Gross_Margin_Pct"] >= nonpromo_gm * 0.8) &
        (promo_campaign_summary["Revenue"] >= promo_revenue_median),

        (promo_campaign_summary["Gross_Margin_Pct"] > 0) &
        (promo_campaign_summary["Gross_Margin_Pct"] < nonpromo_gm * 0.8),

        (promo_campaign_summary["Gross_Margin_Pct"] <= 0) &
        (promo_campaign_summary["Revenue"] >= promo_revenue_median),

        (promo_campaign_summary["Gross_Margin_Pct"] <= 0) &
        (promo_campaign_summary["Revenue"] < promo_revenue_median),
    ],
    [
        "Keep / scale carefully",
        "Reduce depth / tighten guardrails",
        "Replace with bundle / min-order",
        "Stop / deprioritize",
    ],
    default="Review",
)

promo_campaign_summary = promo_campaign_summary.sort_values("Revenue", ascending=False).reset_index(drop=True)

web_traffic["traffic_source"] = web_traffic["traffic_source"].fillna("unknown")

web_traffic["bounce_num"] = web_traffic["bounce_rate"].fillna(0) * web_traffic["sessions"].fillna(0)
web_traffic["duration_num"] = web_traffic["avg_session_duration_sec"].fillna(0) * web_traffic["sessions"].fillna(0)

traffic_daily = (
    web_traffic
    .groupby("date", as_index=False)
    .agg(
        sessions=("sessions", "sum"),
        unique_visitors=("unique_visitors", "sum"),
        page_views=("page_views", "sum"),
        bounce_num=("bounce_num", "sum"),
        duration_num=("duration_num", "sum"),
    )
)

traffic_daily["bounce_rate"] = safe_divide(traffic_daily["bounce_num"], traffic_daily["sessions"])
traffic_daily["avg_session_duration_sec"] = safe_divide(
    traffic_daily["duration_num"],
    traffic_daily["sessions"],
)

daily_traffic_revenue = daily.merge(traffic_daily, on="date", how="left")
daily_traffic_revenue["Revenue_per_Session"] = safe_divide(
    daily_traffic_revenue["Revenue"],
    daily_traffic_revenue["sessions"],
)
daily_traffic_revenue["Orders_per_Session"] = safe_divide(
    daily_traffic_revenue["Orders"],
    daily_traffic_revenue["sessions"],
)

traffic_monthly = (
    daily_traffic_revenue
    .groupby(pd.Grouper(key="date", freq="MS")) # Removed as_index=False to ensure date becomes index
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Orders", "sum"),
        AOV=("AOV", "mean"),
        ARPU=("ARPU", "mean"),
        sessions=("sessions", "sum"),
        unique_visitors=("unique_visitors", "sum"),
        page_views=("page_views", "sum"),
    )
    .reset_index() # Added to convert 'date' index back to a column
)

traffic_monthly["Revenue_per_Session"] = safe_divide(
    traffic_monthly["Revenue"],
    traffic_monthly["sessions"],
)
traffic_monthly["Orders_per_Session"] = safe_divide(
    traffic_monthly["Orders"],
    traffic_monthly["sessions"],
)
traffic_monthly["period"] = traffic_monthly["date"].dt.to_period("M").astype(str)

fact["order_source_norm"] = fact["order_source"].astype(str).str.strip().str.lower()
web_traffic["traffic_source_norm"] = web_traffic["traffic_source"].astype(str).str.strip().str.lower()

source_revenue = (
    fact
    .groupby("order_source_norm", as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        Orders=("order_id", "nunique"),
        Customers=("customer_id", "nunique"),
    )
)

source_traffic = (
    web_traffic
    .groupby("traffic_source_norm", as_index=False)
    .agg(
        Sessions=("sessions", "sum"),
        Unique_Visitors=("unique_visitors", "sum"),
    )
)

source_efficiency = source_revenue.merge(
    source_traffic,
    left_on="order_source_norm",
    right_on="traffic_source_norm",
    how="inner",
)

source_efficiency["Revenue_per_Session"] = safe_divide(
    source_efficiency["Revenue"],
    source_efficiency["Sessions"],
)
source_efficiency["Orders_per_Session"] = safe_divide(
    source_efficiency["Orders"],
    source_efficiency["Sessions"],
)
source_efficiency["Source"] = source_efficiency["order_source_norm"].str.title()
source_efficiency = source_efficiency.sort_values("Revenue_per_Session", ascending=False)

display(promo_summary)
display(promo_campaign_summary.head())
display(traffic_monthly.head())

In [ ]:
# ============================================================
# D1. PROMO VS NON-PROMO SUMMARY
# ============================================================

display_table(
    promo_summary[[
        "promo_group", "Revenue", "Revenue_Share",
        "Orders", "Order_Share", "Customers",
        "Discount", "Discount_Rate", "COGS",
        "Gross_Profit", "Gross_Margin_Pct", "AOV", "ARPU"
    ]],
    money_cols=["Revenue", "Discount", "COGS", "Gross_Profit", "AOV", "ARPU"],
    pct_cols=["Revenue_Share", "Order_Share", "Discount_Rate", "Gross_Margin_Pct"],
    int_cols=["Orders", "Customers"],
)

print("PROMOTION CAMPAIGN TABLE")
display_table(
    promo_campaign_summary[[
        "promo_label", "promo_type_ref", "min_order_value_ref",
        "Revenue", "Revenue_Share", "Discount", "Discount_Rate",
        "Gross_Profit", "Gross_Margin_Pct", "Orders", "AOV",
        "Recommendation"
    ]].head(20),
    money_cols=[
        "min_order_value_ref", "Revenue", "Discount",
        "Gross_Profit", "AOV"
    ],
    pct_cols=["Revenue_Share", "Discount_Rate", "Gross_Margin_Pct"],
    int_cols=["Orders"],
)

In [ ]:
# ============================================================
# D2. BAR CHART - PROMO VS NON-PROMO
# ============================================================

promo_plot = promo_summary.copy()
promo_plot["Revenue_scaled"] = promo_plot["Revenue"] / REVENUE_SCALE

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(
    data=promo_plot,
    x="promo_group",
    y="Revenue_scaled",
    palette=promo_palette,
    ax=axes[0],
)
axes[0].set_title("Revenue", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel(f"Revenue ({REVENUE_UNIT})")

sns.barplot(
    data=promo_plot,
    x="promo_group",
    y="Gross_Margin_Pct",
    palette=promo_palette,
    ax=axes[1],
)
axes[1].set_title("Gross Margin %", fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("GM %")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

sns.barplot(
    data=promo_plot,
    x="promo_group",
    y="Orders",
    palette=promo_palette,
    ax=axes[2],
)
axes[2].set_title("Orders", fontweight="bold")
axes[2].set_xlabel("")
axes[2].set_ylabel("Orders")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# D3. AVERAGE CALENDAR-MONTH PROMO DISCOUNT RATE VS GM%
# ============================================================

promo_month_points = monthly_promo_eff[monthly_promo_eff["promo_group"] == "Promo"].copy()
promo_month_points["month"] = promo_month_points["date"].dt.month

month_avg = (
    promo_month_points
    .groupby("month", as_index=False)
    .agg(
        Avg_Discount_Rate=("Discount_Rate", "mean"),
        Avg_Gross_Margin_Pct=("Gross_Margin_Pct", "mean"),
        Avg_Revenue=("Revenue", "mean"),
        Avg_Orders=("Orders", "mean"),
        Month_Count=("period", "nunique"),
    )
)

month_avg["month_name"] = month_avg["month"].map(lambda m: month_labels[m - 1])
month_avg["Avg_Revenue_scaled"] = month_avg["Avg_Revenue"] / REVENUE_SCALE

display_table(
    month_avg[[
        "month", "month_name", "Avg_Discount_Rate",
        "Avg_Gross_Margin_Pct", "Avg_Revenue",
        "Avg_Orders", "Month_Count"
    ]],
    money_cols=["Avg_Revenue"],
    pct_cols=["Avg_Discount_Rate", "Avg_Gross_Margin_Pct"],
    int_cols=["Avg_Orders", "Month_Count"],
)

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=month_avg,
    x="Avg_Discount_Rate",
    y="Avg_Gross_Margin_Pct",
    size="Avg_Revenue_scaled",
    sizes=(120, 900),
    color=promo_palette["Promo"],
    alpha=0.78,
    edgecolor="black",
    linewidth=0.8,
)

for _, row in month_avg.iterrows():
    plt.text(
        row["Avg_Discount_Rate"] + 0.001,
        row["Avg_Gross_Margin_Pct"] + 0.001,
        row["month_name"],
        fontsize=10,
        weight="bold",
    )

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Promo: Average Calendar-Month Discount Rate vs Gross Margin %", fontsize=16, fontweight="bold")
plt.xlabel("Average Promo Discount Rate by Calendar Month")
plt.ylabel("Average Promo Gross Margin % by Calendar Month")
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# D4. TRAFFIC VS REVENUE
# ============================================================

traffic_plot = traffic_monthly.dropna(subset=["sessions"]).copy()
traffic_plot["Revenue_scaled"] = traffic_plot["Revenue"] / REVENUE_SCALE

fig, ax1 = plt.subplots(figsize=(15, 6))

sns.lineplot(
    data=traffic_plot,
    x="date",
    y="sessions",
    color="#1f77b4",
    linewidth=2.2,
    ax=ax1,
)
ax1.set_ylabel("Sessions", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")
ax1.set_xlabel("Month")

ax2 = ax1.twinx()
sns.lineplot(
    data=traffic_plot,
    x="date",
    y="Revenue_scaled",
    color="#d62728",
    linewidth=2.2,
    ax=ax2,
)
ax2.set_ylabel(f"Revenue ({REVENUE_UNIT})", color="#d62728")
ax2.tick_params(axis="y", labelcolor="#d62728")

plt.title("Monthly Traffic vs Revenue", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

traffic_corr = traffic_plot[["sessions", "Revenue", "Orders", "Revenue_per_Session", "Orders_per_Session"]].corr()
display(traffic_corr)

In [ ]:
# ============================================================
# D5. REVENUE PER SESSION BY SOURCE
# ============================================================

source_plot = source_efficiency.copy()
source_plot["Revenue_per_Session_scaled"] = source_plot["Revenue_per_Session"] / REVENUE_SCALE

display_table(
    source_efficiency[[
        "Source", "Revenue", "Orders", "Customers",
        "Sessions", "Unique_Visitors",
        "Revenue_per_Session", "Orders_per_Session"
    ]],
    money_cols=["Revenue", "Revenue_per_Session"],
    int_cols=["Orders", "Customers", "Sessions", "Unique_Visitors"],
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=source_plot.sort_values("Revenue_per_Session_scaled", ascending=True),
    x="Revenue_per_Session_scaled",
    y="Source",
    color="#72B7B2",
)

plt.title("Revenue per Session by Traffic Source", fontsize=16, fontweight="bold")
plt.xlabel(f"Revenue per Session ({REVENUE_UNIT})")
plt.ylabel("Traffic Source")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# D6. AOV / ARPU TREND
# ============================================================

monetization_plot = monthly[["date", "AOV", "ARPU"]].copy()
monetization_long = monetization_plot.melt(
    id_vars=["date"],
    value_vars=["AOV", "ARPU"],
    var_name="Metric",
    value_name="Value",
)

plt.figure(figsize=(15, 6))
sns.lineplot(
    data=monetization_long,
    x="date",
    y="Value",
    hue="Metric",
    linewidth=2.2,
    palette={
        "AOV": metric_palette["AOV"],
        "ARPU": metric_palette["ARPU"],
    },
)

plt.title("Monthly AOV and ARPU Trend", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Value")
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# D7. ACTION MATRIX + AUTO INSIGHTS
# ============================================================

discount_guardrail = promo_month_points["Discount_Rate"].quantile(0.75)
promo_margin_floor = promo_month_points["Gross_Margin_Pct"].quantile(0.25)

promo_guardrails = pd.DataFrame({
    "Guardrail": [
        "Promo discount rate ceiling",
        "Promo gross margin floor",
        "Revenue per session monitoring line",
    ],
    "Suggested_Threshold": [
        discount_guardrail,
        promo_margin_floor,
        traffic_plot["Revenue_per_Session"].quantile(0.25),
    ],
    "Interpretation": [
        "Above this discount level, promo effectiveness becomes risky",
        "Below this GM level, promo is likely destroying profit",
        "If monetization falls below this level, traffic quality may be weakening",
    ],
})

display_table(
    promo_guardrails,
    pct_cols=["Suggested_Threshold"]
)

print("PROMO ACTION MATRIX")
display_table(
    promo_campaign_summary[[
        "promo_label", "promo_type_ref", "min_order_value_ref",
        "Revenue", "Revenue_Share", "Gross_Margin_Pct",
        "Discount_Rate", "Orders", "AOV", "Recommendation"
    ]].head(25),
    money_cols=["min_order_value_ref", "Revenue", "AOV"],
    pct_cols=["Revenue_Share", "Gross_Margin_Pct", "Discount_Rate"],
    int_cols=["Orders"],
)

promo_vs_nonpromo = promo_summary.set_index("promo_group")
promo_revenue_share = promo_vs_nonpromo.loc["Promo", "Revenue_Share"]
promo_gm = promo_vs_nonpromo.loc["Promo", "Gross_Margin_Pct"]
nonpromo_gm = promo_vs_nonpromo.loc["Non-Promo", "Gross_Margin_Pct"]
sessions_revenue_corr = traffic_plot["sessions"].corr(traffic_plot["Revenue"])
sessions_orders_corr = traffic_plot["sessions"].corr(traffic_plot["Orders"])

print("D. PROMOTION, TRAFFIC & MONETIZATION EFFICIENCY - KEY INSIGHTS")
print("-" * 70)
print(f"Promo revenue share: {promo_revenue_share:.2%}")
print(f"Promo gross margin %: {promo_gm:.2%}")
print(f"Non-promo gross margin %: {nonpromo_gm:.2%}")
print(f"GM gap (Promo - Non-promo): {promo_gm - nonpromo_gm:.2%}")
print(f"Correlation between monthly sessions and revenue: {sessions_revenue_corr:.3f}")
print(f"Correlation between monthly sessions and orders: {sessions_orders_corr:.3f}")
print()
print("Top campaign recommendations:")
for _, r in promo_campaign_summary.head(10).iterrows():
    print(
        f"- {r['promo_label']}: {r['Recommendation']} | "
        f"Revenue Share = {r['Revenue_Share']:.2%} | "
        f"GM % = {r['Gross_Margin_Pct']:.2%} | "
        f"Discount Rate = {r['Discount_Rate']:.2%}"
    )

In [ ]:
# ============================================================
# E0. BUILD INVENTORY DATASETS
# ============================================================


inv = inventory.copy()
prod = products.copy()
fact_e = fact.copy()
ret_e = returns.copy()


inv["product_id"] = inv["product_id"].astype(str)
prod["product_id"] = prod["product_id"].astype(str)
fact_e["product_id"] = fact_e["product_id"].astype(str)
ret_e["product_id"] = ret_e["product_id"].astype(str)


drop_product_attrs = ["product_name", "category", "segment", "size", "color"]
inv = inv.drop(columns=[c for c in drop_product_attrs if c in inv.columns], errors="ignore")


product_lookup = prod[[
    "product_id", "product_name", "category", "segment", "size", "color"
]].drop_duplicates("product_id")

inventory_core = inv.merge(product_lookup, on="product_id", how="left")


inventory_numeric_cols = [
    "stock_on_hand", "units_received", "units_sold", "stockout_days",
    "days_of_supply", "fill_rate", "stockout_flag",
    "overstock_flag", "reorder_flag", "sell_through_rate"
]

for c in inventory_numeric_cols:
    inventory_core[c] = pd.to_numeric(inventory_core[c], errors="coerce").fillna(0)

ret_e["refund_amount"] = pd.to_numeric(ret_e["refund_amount"], errors="coerce").fillna(0)
ret_e["return_quantity"] = pd.to_numeric(ret_e["return_quantity"], errors="coerce").fillna(0)

inventory_core["snapshot_month"] = inventory_core["snapshot_date"].dt.to_period("M").dt.to_timestamp()
inventory_core["year"] = inventory_core["snapshot_date"].dt.year
inventory_core["month"] = inventory_core["snapshot_date"].dt.month
inventory_core["month_name"] = inventory_core["snapshot_date"].dt.month_name()


inventory_core["fill_rate_num"] = inventory_core["fill_rate"] * inventory_core["units_sold"]
inventory_core["sell_through_num"] = inventory_core["sell_through_rate"] * inventory_core["units_sold"]
inventory_core["days_supply_num"] = inventory_core["days_of_supply"] * inventory_core["stock_on_hand"]

product_perf = (
    fact_e
    .groupby("product_id", as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        Gross_Profit=("gross_profit", "sum"),
        Units_Revenue=("quantity", "sum"),
        Orders=("order_id", "nunique")
    )
)

product_perf["Gross_Margin_Pct"] = safe_divide(product_perf["Gross_Profit"], product_perf["Revenue"])
product_perf["Revenue_Share"] = safe_divide(product_perf["Revenue"], product_perf["Revenue"].sum())

category_perf = (
    fact_e
    .groupby("category", as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        Gross_Profit=("gross_profit", "sum")
    )
)

category_perf["Revenue_Share"] = safe_divide(category_perf["Revenue"], category_perf["Revenue"].sum())
category_perf["Gross_Margin_Pct"] = safe_divide(category_perf["Gross_Profit"], category_perf["Revenue"])

product_returns = (
    ret_e
    .groupby("product_id", as_index=False)
    .agg(
        Refund_Amount=("refund_amount", "sum"),
        Return_Units=("return_quantity", "sum")
    )
)

category_inventory = (
    inventory_core
    .groupby("category", as_index=False)
    .agg(
        stockout_days=("stockout_days", "sum"),
        stock_on_hand=("stock_on_hand", "sum"),
        units_received=("units_received", "sum"),
        units_sold=("units_sold", "sum"),
        fill_rate_num=("fill_rate_num", "sum"),
        sell_through_num=("sell_through_num", "sum"),
        days_supply_num=("days_supply_num", "sum"),
        overstock_rate=("overstock_flag", "mean"),
        reorder_rate=("reorder_flag", "mean"),
        stockout_flag_rate=("stockout_flag", "mean")
    )
)

category_inventory["stockout_share"] = safe_divide(
    category_inventory["stockout_days"],
    category_inventory["stockout_days"].sum()
)

category_inventory["units_share"] = safe_divide(
    category_inventory["units_sold"],
    category_inventory["units_sold"].sum()
)

category_inventory["fill_rate_wavg"] = safe_divide(
    category_inventory["fill_rate_num"],
    category_inventory["units_sold"]
)

category_inventory["sell_through_rate_wavg"] = safe_divide(
    category_inventory["sell_through_num"],
    category_inventory["units_sold"]
)

category_inventory["days_of_supply_wavg"] = safe_divide(
    category_inventory["days_supply_num"],
    category_inventory["stock_on_hand"]
)

category_inventory = (
    category_inventory
    .merge(category_perf, on="category", how="left")
    .sort_values("stockout_days", ascending=False)
    .reset_index(drop=True)
)

inventory_monthly_category = (
    inventory_core
    .groupby(["snapshot_month", "category"], as_index=False)
    .agg(
        stockout_days=("stockout_days", "sum"),
        stock_on_hand=("stock_on_hand", "sum"),
        units_sold=("units_sold", "sum"),
        fill_rate_num=("fill_rate_num", "sum"),
        sell_through_num=("sell_through_num", "sum"),
        days_supply_num=("days_supply_num", "sum")
    )
)

inventory_monthly_category["month"] = inventory_monthly_category["snapshot_month"].dt.month
inventory_monthly_category["period"] = inventory_monthly_category["snapshot_month"].dt.to_period("M").astype(str)

inventory_monthly_category["fill_rate_wavg"] = safe_divide(
    inventory_monthly_category["fill_rate_num"],
    inventory_monthly_category["units_sold"]
)

inventory_monthly_category["sell_through_rate_wavg"] = safe_divide(
    inventory_monthly_category["sell_through_num"],
    inventory_monthly_category["units_sold"]
)

inventory_monthly_category["days_of_supply_wavg"] = safe_divide(
    inventory_monthly_category["days_supply_num"],
    inventory_monthly_category["stock_on_hand"]
)


sku_inventory = (
    inventory_core
    .groupby(
        ["product_id", "product_name", "category", "segment", "size", "color"],
        as_index=False
    )
    .agg(
        stockout_days=("stockout_days", "sum"),
        stock_on_hand=("stock_on_hand", "sum"),
        units_received=("units_received", "sum"),
        units_sold=("units_sold", "sum"),
        fill_rate_num=("fill_rate_num", "sum"),
        sell_through_num=("sell_through_num", "sum"),
        days_supply_num=("days_supply_num", "sum"),
        overstock_rate=("overstock_flag", "mean"),
        reorder_rate=("reorder_flag", "mean")
    )
)

sku_inventory["fill_rate_wavg"] = safe_divide(sku_inventory["fill_rate_num"], sku_inventory["units_sold"])
sku_inventory["sell_through_rate_wavg"] = safe_divide(sku_inventory["sell_through_num"], sku_inventory["units_sold"])
sku_inventory["days_of_supply_wavg"] = safe_divide(sku_inventory["days_supply_num"], sku_inventory["stock_on_hand"])

sku_inventory = (
    sku_inventory
    .merge(product_perf, on="product_id", how="left")
    .merge(product_returns, on="product_id", how="left")
)

sku_inventory[["Refund_Amount", "Return_Units"]] = sku_inventory[["Refund_Amount", "Return_Units"]].fillna(0)
sku_inventory["Return_Rate"] = safe_divide(sku_inventory["Return_Units"], sku_inventory["units_sold"])


if "planning_calendar" in globals():
    peak_month_nums = sorted(
        planning_calendar.loc[
            planning_calendar["Seasonality_Class"].isin(["Strong peak", "Mild peak"]),
            "month"
        ].unique().tolist()
    )
else:
    peak_month_nums = []

if len(peak_month_nums) == 0 and "seasonality" in globals():
    peak_month_nums = (
        seasonality
        .sort_values("Seasonality_Index", ascending=False)
        .head(3)["month"]
        .tolist()
    )

if len(peak_month_nums) == 0:
    peak_month_nums = [4, 5, 6]

peak_inventory = inventory_core[inventory_core["month"].isin(peak_month_nums)]

peak_sku = (
    peak_inventory
    .groupby("product_id", as_index=False)
    .agg(
        peak_stockout_days=("stockout_days", "sum"),
        peak_units_sold=("units_sold", "sum"),
        peak_stock_on_hand=("stock_on_hand", "sum"),
        peak_fill_rate_num=("fill_rate_num", "sum"),
        peak_sell_through_num=("sell_through_num", "sum"),
        peak_days_supply_num=("days_supply_num", "sum")
    )
)

peak_sku["peak_fill_rate"] = safe_divide(peak_sku["peak_fill_rate_num"], peak_sku["peak_units_sold"])
peak_sku["peak_sell_through_rate"] = safe_divide(peak_sku["peak_sell_through_num"], peak_sku["peak_units_sold"])
peak_sku["peak_days_of_supply"] = safe_divide(peak_sku["peak_days_supply_num"], peak_sku["peak_stock_on_hand"])

sku_priority = sku_inventory.merge(
    peak_sku[[
        "product_id", "peak_stockout_days", "peak_units_sold",
        "peak_fill_rate", "peak_sell_through_rate", "peak_days_of_supply"
    ]],
    on="product_id",
    how="left"
)

fill_zero_cols = [
    "peak_stockout_days", "peak_units_sold",
    "peak_fill_rate", "peak_sell_through_rate", "peak_days_of_supply"
]

sku_priority[fill_zero_cols] = sku_priority[fill_zero_cols].fillna(0)


def pct_rank(s):
    return s.rank(pct=True, method="average").fillna(0)

sku_priority["stockout_score"] = pct_rank(sku_priority["peak_stockout_days"])
sku_priority["demand_score"] = pct_rank(sku_priority["peak_units_sold"])
sku_priority["margin_score"] = pct_rank(sku_priority["Gross_Margin_Pct"].fillna(0))
sku_priority["low_fill_score"] = 1 - pct_rank(sku_priority["peak_fill_rate"])
sku_priority["low_supply_score"] = 1 - pct_rank(sku_priority["peak_days_of_supply"])
sku_priority["return_penalty"] = pct_rank(sku_priority["Return_Rate"].fillna(0))

sku_priority["Priority_Score"] = 100 * (
    0.30 * sku_priority["stockout_score"] +
    0.25 * sku_priority["demand_score"] +
    0.15 * sku_priority["margin_score"] +
    0.15 * sku_priority["low_fill_score"] +
    0.10 * sku_priority["low_supply_score"] -
    0.05 * sku_priority["return_penalty"]
)

sku_priority["Suggested_Safety_Stock_Units"] = np.ceil(
    safe_divide(sku_priority["peak_units_sold"], max(len(peak_month_nums), 1)) * 0.25
)

priority_cutoff = sku_priority["Priority_Score"].quantile(0.85)
monitor_cutoff = sku_priority["Priority_Score"].quantile(0.60)
overstock_cutoff = sku_priority["overstock_rate"].quantile(0.75)
sellthrough_median = sku_priority["sell_through_rate_wavg"].median()
peak_supply_median = sku_priority["peak_days_of_supply"].median()

sku_priority["Suggested_Action"] = np.select(
    [
        (sku_priority["Priority_Score"] >= priority_cutoff) &
        (sku_priority["peak_days_of_supply"] <= peak_supply_median),

        (sku_priority["overstock_rate"] >= overstock_cutoff) &
        (sku_priority["sell_through_rate_wavg"] <= sellthrough_median),

        sku_priority["Priority_Score"] >= monitor_cutoff
    ],
    [
        "Reorder / build safety stock",
        "Reduce inbound / clear stock carefully",
        "Monitor closely"
    ],
    default="Normal"
)

sku_priority = sku_priority.sort_values("Priority_Score", ascending=False).reset_index(drop=True)

print("Inventory datasets created successfully.")
print("Peak months used:", peak_month_nums)

display(category_inventory.head())
display(sku_priority.head())

In [ ]:
# ============================================================
# E1. INVENTORY SUMMARY TABLES
# ============================================================

weighted_fill_rate = safe_divide(
    inventory_core["fill_rate_num"].sum(),
    inventory_core["units_sold"].sum()
)

weighted_sell_through = safe_divide(
    inventory_core["sell_through_num"].sum(),
    inventory_core["units_sold"].sum()
)

weighted_days_supply = safe_divide(
    inventory_core["days_supply_num"].sum(),
    inventory_core["stock_on_hand"].sum()
)

inventory_overview = pd.DataFrame({
    "Metric": [
        "Total stockout days",
        "Weighted fill rate",
        "Weighted sell-through rate",
        "Weighted days of supply",
        "Top 3 stockout categories"
    ],
    "Value": [
        category_inventory["stockout_days"].sum(),
        weighted_fill_rate,
        weighted_sell_through,
        weighted_days_supply,
        ", ".join(category_inventory.head(3)["category"].astype(str))
    ]
})

display(inventory_overview)

display_table(
    category_inventory[[
        "category", "stockout_days", "stockout_share",
        "units_sold", "units_share",
        "fill_rate_wavg", "sell_through_rate_wavg",
        "days_of_supply_wavg", "overstock_rate", "reorder_rate",
        "Revenue_Share", "Gross_Margin_Pct"
    ]],
    pct_cols=[
        "stockout_share", "units_share",
        "fill_rate_wavg", "sell_through_rate_wavg",
        "overstock_rate", "reorder_rate",
        "Revenue_Share", "Gross_Margin_Pct"
    ],
    int_cols=["stockout_days", "units_sold"]
)

In [ ]:
# ============================================================
# E2. HEATMAP - CATEGORY x CALENDAR MONTH STOCKOUT DAYS
# ============================================================

stockout_heatmap = (
    inventory_monthly_category
    .groupby(["category", "month"], as_index=False)
    .agg(avg_stockout_days=("stockout_days", "mean"))
    .pivot(index="category", columns="month", values="avg_stockout_days")
    .reindex(index=category_inventory["category"], columns=month_order)
)

plt.figure(figsize=(14, 6))

sns.heatmap(
    stockout_heatmap,
    cmap="YlOrRd",
    annot=True,
    fmt=".0f",
    linewidths=0.4,
    linecolor="white",
    mask=stockout_heatmap.isna(),
    cbar_kws={"label": "Average Stockout Days"}
)

plt.title("Average Stockout Days by Category and Calendar Month", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Category")
plt.xticks(ticks=np.arange(12) + 0.5, labels=month_labels, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# E3. MONTHLY STOCKOUT DAYS BY CATEGORY
# ============================================================

e_palette = {
    cat: category_palette.get(cat, "#4C78A8")
    for cat in inventory_monthly_category["category"].dropna().unique()
}

plt.figure(figsize=(15, 6))

sns.lineplot(
    data=inventory_monthly_category,
    x="snapshot_month",
    y="stockout_days",
    hue="category",
    linewidth=2.1,
    palette=e_palette
)

plt.title("Monthly Stockout Days by Category", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Stockout Days")
plt.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# E4. SELL-THROUGH RATE VS DAYS OF SUPPLY
# ============================================================

e4_df = category_inventory.dropna(
    subset=["days_of_supply_wavg", "sell_through_rate_wavg"]
).copy()

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=e4_df,
    x="days_of_supply_wavg",
    y="sell_through_rate_wavg",
    hue="category",
    size="stockout_days",
    palette={cat: category_palette.get(cat, "#4C78A8") for cat in e4_df["category"]},
    sizes=(200, 2500),
    alpha=0.75,
    edgecolor="black",
    linewidth=0.7
)

for _, r in e4_df.iterrows():
    plt.text(
        r["days_of_supply_wavg"] + 0.2,
        r["sell_through_rate_wavg"] + 0.002,
        r["category"],
        fontsize=10,
        weight="bold"
    )

plt.title("Sell-through Rate vs Days of Supply", fontsize=16, fontweight="bold")
plt.xlabel("Weighted Days of Supply")
plt.ylabel("Weighted Sell-through Rate")
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.legend(title="Category / Stockout Days", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# E5. INVENTORY QUADRANT
# ============================================================

quadrant_df = category_inventory.copy()

units_median = quadrant_df["units_share"].median()
stockout_median = quadrant_df["stockout_share"].median()

quadrant_df["Quadrant"] = np.select(
    [
        (quadrant_df["units_share"] >= units_median) &
        (quadrant_df["stockout_share"] >= stockout_median),

        (quadrant_df["units_share"] >= units_median) &
        (quadrant_df["stockout_share"] < stockout_median),

        (quadrant_df["units_share"] < units_median) &
        (quadrant_df["stockout_share"] >= stockout_median),

        (quadrant_df["units_share"] < units_median) &
        (quadrant_df["stockout_share"] < stockout_median),
    ],
    [
        "High demand / High stockout",
        "High demand / Low stockout",
        "Low demand / High stockout",
        "Low demand / Low stockout",
    ],
    default="Review"
)

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=quadrant_df,
    x="units_share",
    y="stockout_share",
    hue="category",
    size="Revenue_Share",
    palette={cat: category_palette.get(cat, "#4C78A8") for cat in quadrant_df["category"]},
    sizes=(250, 2500),
    alpha=0.8,
    edgecolor="black",
    linewidth=0.8
)

for _, r in quadrant_df.iterrows():
    plt.text(
        r["units_share"] + 0.002,
        r["stockout_share"] + 0.002,
        r["category"],
        fontsize=10,
        weight="bold"
    )

plt.axvline(units_median, color="gray", linestyle="--", linewidth=1)
plt.axhline(stockout_median, color="gray", linestyle="--", linewidth=1)

plt.title("Inventory Quadrant: Demand vs Stockout", fontsize=16, fontweight="bold")
plt.xlabel("Units Share")
plt.ylabel("Stockout Share")
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.legend(title="Category / Revenue Share", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

display_table(
    quadrant_df[[
        "category", "units_share", "stockout_share", "Revenue_Share",
        "fill_rate_wavg", "days_of_supply_wavg",
        "sell_through_rate_wavg", "Quadrant"
    ]],
    pct_cols=[
        "units_share", "stockout_share", "Revenue_Share",
        "fill_rate_wavg", "sell_through_rate_wavg"
    ]
)

In [ ]:
# ============================================================
# E6. SKU PRIORITY TABLE + AUTO INSIGHTS
# ============================================================

reorder_table = (
    sku_priority[sku_priority["Suggested_Action"] == "Reorder / build safety stock"]
    .sort_values("Priority_Score", ascending=False)
    .head(20)
)

reduce_table = (
    sku_priority[sku_priority["Suggested_Action"] == "Reduce inbound / clear stock carefully"]
    .sort_values("Priority_Score", ascending=False)
    .head(20)
)

print("REORDER PRIORITY TABLE")
display_table(
    reorder_table[[
        "product_id", "product_name", "category", "size", "color",
        "Priority_Score", "peak_stockout_days", "peak_units_sold",
        "peak_fill_rate", "peak_days_of_supply",
        "Gross_Margin_Pct", "Return_Rate",
        "Suggested_Safety_Stock_Units"
    ]],
    pct_cols=["peak_fill_rate", "Gross_Margin_Pct", "Return_Rate"],
    int_cols=[
        "Priority_Score", "peak_stockout_days",
        "peak_units_sold", "Suggested_Safety_Stock_Units"
    ]
)

print("REDUCE INBOUND / CLEAR STOCK TABLE")
display_table(
    reduce_table[[
        "product_id", "product_name", "category", "size", "color",
        "Priority_Score", "overstock_rate", "sell_through_rate_wavg",
        "days_of_supply_wavg", "Gross_Margin_Pct",
        "Return_Rate", "Suggested_Action"
    ]],
    pct_cols=[
        "overstock_rate", "sell_through_rate_wavg",
        "Gross_Margin_Pct", "Return_Rate"
    ],
    int_cols=["Priority_Score"]
)

top_stockout_categories = category_inventory.head(3)
top_reorder_skus = reorder_table.head(5)

print("E. INVENTORY & FULFILLMENT READINESS - KEY INSIGHTS")
print("-" * 70)
print(f"Peak-season months used for risk scoring: {peak_month_nums}")
print(f"Weighted fill rate: {weighted_fill_rate:.2%}")
print(f"Weighted sell-through rate: {weighted_sell_through:.2%}")
print(f"Weighted days of supply: {weighted_days_supply:.2f}")

print("\nTop stockout categories:")
for _, r in top_stockout_categories.iterrows():
    print(
        f"- {r['category']}: Stockout Share = {r['stockout_share']:.2%} | "
        f"Units Share = {r['units_share']:.2%} | "
        f"Revenue Share = {r['Revenue_Share']:.2%}"
    )

print("\nTop reorder candidates:")
for _, r in top_reorder_skus.iterrows():
    print(
        f"- {r['product_name']} ({r['category']} / {r['size']} / {r['color']}): "
        f"Priority Score = {r['Priority_Score']:.1f} | "
        f"Peak Stockout Days = {r['peak_stockout_days']:.0f} | "
        f"Suggested Safety Stock = {r['Suggested_Safety_Stock_Units']:.0f}"
    )

In [ ]:
# ============================================================
# F0. PREPARE CUSTOMER / REGION DATASETS
# ============================================================

fact_f = fact.copy()
orders_f = orders.copy()
customers_f = customers.copy()
geo_f = geography.copy()
returns_f = returns.copy()
shipments_f = shipments.copy()

for df in [fact_f, orders_f, returns_f, shipments_f]:
    if "order_id" in df.columns:
        df["order_id"] = df["order_id"].astype(str)

for df in [fact_f, orders_f, customers_f]:
    if "customer_id" in df.columns:
        df["customer_id"] = df["customer_id"].astype(str)

for df in [fact_f, orders_f, customers_f, geo_f]:
    if "zip" in df.columns:
        df["zip"] = df["zip"].astype(str)

order_value = (
    fact_f
    .groupby("order_id", as_index=False)
    .agg(
        Revenue=("item_revenue", "sum"),
        COGS=("cogs_total", "sum"),
        Gross_Profit=("gross_profit", "sum"),
        Discount=("discount_amount", "sum"),
        Units=("quantity", "sum")
    )
)

order_value["Gross_Margin_Pct"] = safe_divide(order_value["Gross_Profit"], order_value["Revenue"])

customer_attrs = customers_f[[
    "customer_id", "gender", "age_group", "acquisition_channel"
]].copy()

for c in ["gender", "age_group", "acquisition_channel"]:
    customer_attrs[c] = customer_attrs[c].fillna("Unknown").astype(str)

geo_attrs = geo_f[["zip", "city", "region", "district"]].drop_duplicates("zip").copy()

order_base = (
    orders_f
    .merge(order_value, on="order_id", how="left")
    .merge(customer_attrs, on="customer_id", how="left")
    .merge(geo_attrs, on="zip", how="left")
)

for c in ["Revenue", "COGS", "Gross_Profit", "Discount", "Units"]:
    order_base[c] = order_base[c].fillna(0)

for c in ["gender", "age_group", "acquisition_channel", "region", "city", "district"]:
    order_base[c] = order_base[c].fillna("Unknown").astype(str)

order_base["order_month"] = order_base["order_date"].dt.to_period("M").dt.to_timestamp()

returns_enriched_f = (
    returns_f
    .merge(
        orders_f[["order_id", "order_date", "customer_id", "zip"]],
        on="order_id",
        how="left"
    )
    .merge(customer_attrs, on="customer_id", how="left")
    .merge(geo_attrs, on="zip", how="left")
)

returns_enriched_f["refund_amount"] = pd.to_numeric(
    returns_enriched_f["refund_amount"],
    errors="coerce"
).fillna(0)

returns_enriched_f["return_quantity"] = pd.to_numeric(
    returns_enriched_f["return_quantity"],
    errors="coerce"
).fillna(0)

for c in ["gender", "age_group", "acquisition_channel", "region"]:
    returns_enriched_f[c] = returns_enriched_f[c].fillna("Unknown").astype(str)

delivery_base = (
    shipments_f
    .merge(
        orders_f[["order_id", "order_date", "customer_id", "zip"]],
        on="order_id",
        how="left"
    )
    .merge(customer_attrs, on="customer_id", how="left")
    .merge(geo_attrs, on="zip", how="left")
)

delivery_base["delivery_lead_days"] = (
    delivery_base["delivery_date"] - delivery_base["order_date"]
).dt.days

delivery_base = delivery_base[
    delivery_base["delivery_lead_days"].notna() &
    (delivery_base["delivery_lead_days"] >= 0)
].copy()

delivery_base["region"] = delivery_base["region"].fillna("Unknown").astype(str)

customer_orders = (
    order_base[["customer_id", "order_id", "order_date", "region", "age_group", "acquisition_channel"]]
    .drop_duplicates(["customer_id", "order_id"])
    .sort_values(["customer_id", "order_date"])
)

first_order = (
    customer_orders
    .drop_duplicates("customer_id", keep="first")
    .rename(columns={
        "order_date": "first_order_date",
        "region": "first_region",
        "age_group": "first_age_group",
        "acquisition_channel": "first_acquisition_channel"
    })[[
        "customer_id", "first_order_date",
        "first_region", "first_age_group", "first_acquisition_channel"
    ]]
)

customer_order_counts = (
    customer_orders
    .groupby("customer_id", as_index=False)
    .agg(total_orders=("order_id", "nunique"))
)

customer_repeat_events = (
    customer_orders
    .merge(first_order[["customer_id", "first_order_date"]], on="customer_id", how="left")
)

customer_repeat_events["days_since_first_order"] = (
    customer_repeat_events["order_date"] - customer_repeat_events["first_order_date"]
).dt.days

customer_repeat_flags = (
    customer_repeat_events
    .groupby("customer_id", as_index=False)
    .agg(
        repeat_30d=("days_since_first_order", lambda x: ((x > 0) & (x <= 30)).any()),
        repeat_90d=("days_since_first_order", lambda x: ((x > 0) & (x <= 90)).any())
    )
)

customer_profile = (
    first_order
    .merge(customer_order_counts, on="customer_id", how="left")
    .merge(customer_repeat_flags, on="customer_id", how="left")
)

customer_profile["is_repeat_customer"] = customer_profile["total_orders"] > 1
customer_profile["first_purchase_month"] = customer_profile["first_order_date"].dt.to_period("M").dt.to_timestamp()

print("Customer and regional datasets created successfully.")
display(order_base.head())
display(customer_profile.head())

In [ ]:
# ============================================================
# F1. GROUP SUMMARY TABLES
# ============================================================

def build_group_summary(group_col, first_group_col=None):
    first_group_col = first_group_col or group_col

    sales_part = (
        order_base
        .groupby(group_col, as_index=False)
        .agg(
            Revenue=("Revenue", "sum"),
            Gross_Profit=("Gross_Profit", "sum"),
            Orders=("order_id", "nunique"),
            Active_Customers=("customer_id", "nunique"),
            Units=("Units", "sum")
        )
    )

    sales_part["Revenue_Share"] = safe_divide(sales_part["Revenue"], sales_part["Revenue"].sum())
    sales_part["Customer_Share"] = safe_divide(
        sales_part["Active_Customers"],
        sales_part["Active_Customers"].sum()
    )
    sales_part["Revenue_per_Customer"] = safe_divide(
        sales_part["Revenue"],
        sales_part["Active_Customers"]
    )
    sales_part["AOV"] = safe_divide(sales_part["Revenue"], sales_part["Orders"])
    sales_part["Gross_Margin_Pct"] = safe_divide(
        sales_part["Gross_Profit"],
        sales_part["Revenue"]
    )

    refund_part = (
        returns_enriched_f
        .groupby(group_col, as_index=False)
        .agg(
            Refund_Amount=("refund_amount", "sum"),
            Return_Units=("return_quantity", "sum"),
            Return_Orders=("order_id", "nunique")
        )
    )

    repeat_group = (
        customer_profile
        .rename(columns={first_group_col: group_col})
        .groupby(group_col, as_index=False)
        .agg(
            Customers_First_Group=("customer_id", "nunique"),
            Repeat_Customers=("is_repeat_customer", "sum"),
            Repeat_30d_Customers=("repeat_30d", "sum"),
            Repeat_90d_Customers=("repeat_90d", "sum")
        )
    )

    out = (
        sales_part
        .merge(refund_part, on=group_col, how="left")
        .merge(repeat_group, on=group_col, how="left")
    )

    for c in [
        "Refund_Amount", "Return_Units", "Return_Orders",
        "Customers_First_Group", "Repeat_Customers",
        "Repeat_30d_Customers", "Repeat_90d_Customers"
    ]:
        out[c] = out[c].fillna(0)

    out["Refund_to_Revenue"] = safe_divide(out["Refund_Amount"], out["Revenue"])
    out["Repeat_Purchase_Rate"] = safe_divide(
        out["Repeat_Customers"],
        out["Customers_First_Group"]
    )
    out["Repeat_30d_Rate"] = safe_divide(
        out["Repeat_30d_Customers"],
        out["Customers_First_Group"]
    )
    out["Repeat_90d_Rate"] = safe_divide(
        out["Repeat_90d_Customers"],
        out["Customers_First_Group"]
    )

    return out.sort_values("Revenue", ascending=False).reset_index(drop=True)

region_summary = build_group_summary("region", "first_region")
age_summary = build_group_summary("age_group", "first_age_group")
acquisition_summary = build_group_summary("acquisition_channel", "first_acquisition_channel")

delivery_region = (
    delivery_base
    .groupby("region", as_index=False)
    .agg(
        Avg_Delivery_Lead_Days=("delivery_lead_days", "mean"),
        Median_Delivery_Lead_Days=("delivery_lead_days", "median"),
        Delivered_Orders=("order_id", "nunique")
    )
)

region_summary = region_summary.merge(delivery_region, on="region", how="left")

print("REGION SUMMARY")
display_table(
    region_summary,
    money_cols=["Revenue", "Gross_Profit", "Revenue_per_Customer", "AOV", "Refund_Amount"],
    pct_cols=[
        "Revenue_Share", "Customer_Share", "Gross_Margin_Pct",
        "Refund_to_Revenue", "Repeat_Purchase_Rate",
        "Repeat_30d_Rate", "Repeat_90d_Rate"
    ],
    int_cols=[
        "Orders", "Active_Customers", "Units", "Return_Orders",
        "Customers_First_Group", "Repeat_Customers", "Delivered_Orders"
    ]
)

print("AGE GROUP SUMMARY")
display_table(
    age_summary,
    money_cols=["Revenue", "Gross_Profit", "Revenue_per_Customer", "AOV", "Refund_Amount"],
    pct_cols=[
        "Revenue_Share", "Customer_Share", "Gross_Margin_Pct",
        "Refund_to_Revenue", "Repeat_Purchase_Rate",
        "Repeat_30d_Rate", "Repeat_90d_Rate"
    ],
    int_cols=[
        "Orders", "Active_Customers", "Units", "Return_Orders",
        "Customers_First_Group", "Repeat_Customers"
    ]
)

print("ACQUISITION CHANNEL SUMMARY")
display_table(
    acquisition_summary,
    money_cols=["Revenue", "Gross_Profit", "Revenue_per_Customer", "AOV", "Refund_Amount"],
    pct_cols=[
        "Revenue_Share", "Customer_Share", "Gross_Margin_Pct",
        "Refund_to_Revenue", "Repeat_Purchase_Rate",
        "Repeat_30d_Rate", "Repeat_90d_Rate"
    ],
    int_cols=[
        "Orders", "Active_Customers", "Units", "Return_Orders",
        "Customers_First_Group", "Repeat_Customers"
    ]
)

In [ ]:
# ============================================================
# F2. REGION CUSTOMER SHARE + REVENUE PER CUSTOMER
# ============================================================

region_order = region_summary.sort_values("Revenue", ascending=False)["region"].tolist()
region_palette = {
    region: color
    for region, color in zip(region_order, sns.color_palette("tab10", n_colors=len(region_order)))
}

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

sns.barplot(
    data=region_summary,
    x="region",
    y="Customer_Share",
    order=region_order,
    hue="region",
    palette=region_palette,
    dodge=False,
    legend=False,
    ax=axes[0]
)

axes[0].set_title("Customer Share by Region", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Region")
axes[0].set_ylabel("Customer Share")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

for patch in axes[0].patches:
    h = patch.get_height()
    if np.isfinite(h):
        axes[0].text(
            patch.get_x() + patch.get_width() / 2,
            h,
            f"{h:.1%}",
            ha="center",
            va="bottom",
            fontsize=10
        )

sns.barplot(
    data=region_summary,
    x="region",
    y="Revenue_per_Customer",
    order=region_order,
    hue="region",
    palette=region_palette,
    dodge=False,
    legend=False,
    ax=axes[1]
)

axes[1].set_title("Revenue per Customer by Region", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Revenue per Customer")

for patch in axes[1].patches:
    h = patch.get_height()
    if np.isfinite(h):
        axes[1].text(
            patch.get_x() + patch.get_width() / 2,
            h,
            f"{h:,.0f}",
            ha="center",
            va="bottom",
            fontsize=10
        )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# F3. REGION MATRIX
# ============================================================

region_matrix = region_summary.copy()

customer_share_median = region_matrix["Customer_Share"].median()
rpc_median = region_matrix["Revenue_per_Customer"].median()

region_matrix["Market_Type"] = np.select(
    [
        (region_matrix["Customer_Share"] >= customer_share_median) &
        (region_matrix["Revenue_per_Customer"] >= rpc_median),

        (region_matrix["Customer_Share"] >= customer_share_median) &
        (region_matrix["Revenue_per_Customer"] < rpc_median),

        (region_matrix["Customer_Share"] < customer_share_median) &
        (region_matrix["Revenue_per_Customer"] >= rpc_median),

        (region_matrix["Customer_Share"] < customer_share_median) &
        (region_matrix["Revenue_per_Customer"] < rpc_median),
    ],
    [
        "Large & high-value",
        "Large but low-value",
        "Small but high-value",
        "Small & low-value",
    ],
    default="Review"
)

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=region_matrix,
    x="Customer_Share",
    y="Revenue_per_Customer",
    size="Revenue_Share",
    hue="region",
    palette=region_palette,
    sizes=(250, 2500),
    alpha=0.8,
    edgecolor="black",
    linewidth=0.8
)

for _, r in region_matrix.iterrows():
    plt.text(
        r["Customer_Share"] + 0.002,
        r["Revenue_per_Customer"],
        r["region"],
        fontsize=10,
        weight="bold"
    )

plt.axvline(customer_share_median, color="gray", linestyle="--", linewidth=1)
plt.axhline(rpc_median, color="gray", linestyle="--", linewidth=1)

plt.title("Regional Sustainability Matrix", fontsize=16, fontweight="bold")
plt.xlabel("Customer Share")
plt.ylabel("Revenue per Customer")
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.legend(title="Region / Revenue Share", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

display_table(
    region_matrix[[
        "region", "Customer_Share", "Revenue_Share",
        "Revenue_per_Customer", "AOV", "Gross_Margin_Pct",
        "Repeat_Purchase_Rate", "Refund_to_Revenue",
        "Avg_Delivery_Lead_Days", "Market_Type"
    ]],
    money_cols=["Revenue_per_Customer", "AOV"],
    pct_cols=[
        "Customer_Share", "Revenue_Share", "Gross_Margin_Pct",
        "Repeat_Purchase_Rate", "Refund_to_Revenue"
    ]
)

In [ ]:
# ============================================================
# F4. COHORT REPEAT RATE
# ============================================================

cohort_repeat = (
    customer_profile
    .groupby("first_purchase_month", as_index=False)
    .agg(
        Customers=("customer_id", "nunique"),
        Repeat_Customers=("is_repeat_customer", "sum"),
        Repeat_30d_Customers=("repeat_30d", "sum"),
        Repeat_90d_Customers=("repeat_90d", "sum")
    )
)

cohort_repeat["Repeat_Purchase_Rate"] = safe_divide(
    cohort_repeat["Repeat_Customers"],
    cohort_repeat["Customers"]
)

cohort_repeat["Repeat_30d_Rate"] = safe_divide(
    cohort_repeat["Repeat_30d_Customers"],
    cohort_repeat["Customers"]
)

cohort_repeat["Repeat_90d_Rate"] = safe_divide(
    cohort_repeat["Repeat_90d_Customers"],
    cohort_repeat["Customers"]
)

cohort_plot = cohort_repeat[cohort_repeat["Customers"] >= 30].copy()

cohort_long = cohort_plot.melt(
    id_vars=["first_purchase_month", "Customers"],
    value_vars=["Repeat_Purchase_Rate", "Repeat_30d_Rate", "Repeat_90d_Rate"],
    var_name="Metric",
    value_name="Repeat_Rate"
)

cohort_long["Metric"] = cohort_long["Metric"].replace({
    "Repeat_Purchase_Rate": "Repeat Purchase Rate",
    "Repeat_30d_Rate": "30-day Repeat Rate",
    "Repeat_90d_Rate": "90-day Repeat Rate"
})

plt.figure(figsize=(15, 6))

sns.lineplot(
    data=cohort_long,
    x="first_purchase_month",
    y="Repeat_Rate",
    hue="Metric",
    linewidth=2.1,
    palette={
        "Repeat Purchase Rate": "#1f77b4",
        "30-day Repeat Rate": "#ff7f0e",
        "90-day Repeat Rate": "#2ca02c"
    }
)

plt.title("Cohort Repeat Rate by First Purchase Month", fontsize=16, fontweight="bold")
plt.xlabel("First Purchase Month")
plt.ylabel("Repeat Rate")
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

display_table(
    cohort_repeat.tail(24),
    pct_cols=["Repeat_Purchase_Rate", "Repeat_30d_Rate", "Repeat_90d_Rate"],
    int_cols=["Customers", "Repeat_Customers", "Repeat_30d_Customers", "Repeat_90d_Customers"]
)

In [ ]:
# ============================================================
# F5. AOV / GM BY AGE GROUP AND ACQUISITION CHANNEL
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 11))

age_order = age_summary.sort_values("Revenue", ascending=False)["age_group"].tolist()
acq_order = acquisition_summary.sort_values("Revenue", ascending=False)["acquisition_channel"].tolist()

sns.barplot(
    data=age_summary,
    x="age_group",
    y="AOV",
    order=age_order,
    color="#4C78A8",
    ax=axes[0, 0]
)
axes[0, 0].set_title("AOV by Age Group", fontweight="bold")
axes[0, 0].set_xlabel("Age Group")
axes[0, 0].set_ylabel("AOV")

sns.barplot(
    data=age_summary,
    x="age_group",
    y="Gross_Margin_Pct",
    order=age_order,
    color="#72B7B2",
    ax=axes[0, 1]
)
axes[0, 1].set_title("Gross Margin % by Age Group", fontweight="bold")
axes[0, 1].set_xlabel("Age Group")
axes[0, 1].set_ylabel("Gross Margin %")
axes[0, 1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

sns.barplot(
    data=acquisition_summary,
    x="acquisition_channel",
    y="AOV",
    order=acq_order,
    color="#F58518",
    ax=axes[1, 0]
)
axes[1, 0].set_title("AOV by Acquisition Channel", fontweight="bold")
axes[1, 0].set_xlabel("Acquisition Channel")
axes[1, 0].set_ylabel("AOV")
axes[1, 0].tick_params(axis="x", rotation=20)

sns.barplot(
    data=acquisition_summary,
    x="acquisition_channel",
    y="Gross_Margin_Pct",
    order=acq_order,
    color="#54A24B",
    ax=axes[1, 1]
)
axes[1, 1].set_title("Gross Margin % by Acquisition Channel", fontweight="bold")
axes[1, 1].set_xlabel("Acquisition Channel")
axes[1, 1].set_ylabel("Gross Margin %")
axes[1, 1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
axes[1, 1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# F6. ACTION MATRIX + AUTO INSIGHTS
# ============================================================

region_action = region_matrix.copy()

region_action["Recommended_Action"] = np.select(
    [
        region_action["Market_Type"].eq("Large & high-value"),
        region_action["Market_Type"].eq("Small but high-value"),
        region_action["Market_Type"].eq("Large but low-value"),
        region_action["Refund_to_Revenue"] >= region_action["Refund_to_Revenue"].quantile(0.75)
    ],
    [
        "Protect retention, maintain service level, prepare logistics capacity",
        "Test growth campaign with controlled inventory and logistics",
        "Improve AOV/ARPU before aggressive scaling",
        "Audit returns, delivery promise, and product-market fit"
    ],
    default="Maintain and monitor"
)

print("REGIONAL ACTION MATRIX")
display_table(
    region_action[[
        "region", "Market_Type", "Recommended_Action",
        "Customer_Share", "Revenue_Share", "Revenue_per_Customer",
        "AOV", "Gross_Margin_Pct", "Repeat_Purchase_Rate",
        "Refund_to_Revenue", "Avg_Delivery_Lead_Days"
    ]],
    money_cols=["Revenue_per_Customer", "AOV"],
    pct_cols=[
        "Customer_Share", "Revenue_Share", "Gross_Margin_Pct",
        "Repeat_Purchase_Rate", "Refund_to_Revenue"
    ]
)

best_rpc_region = region_summary.loc[region_summary["Revenue_per_Customer"].idxmax()]
largest_customer_region = region_summary.loc[region_summary["Customer_Share"].idxmax()]
best_repeat_region = region_summary.loc[region_summary["Repeat_Purchase_Rate"].idxmax()]
highest_refund_region = region_summary.loc[region_summary["Refund_to_Revenue"].idxmax()]

print("F. CUSTOMER & REGIONAL SUSTAINABILITY - KEY INSIGHTS")
print("-" * 70)
print(
    f"Largest customer region: {largest_customer_region['region']} | "
    f"Customer Share = {largest_customer_region['Customer_Share']:.2%} | "
    f"Revenue per Customer = {largest_customer_region['Revenue_per_Customer']:,.0f}"
)
print(
    f"Highest value region: {best_rpc_region['region']} | "
    f"Customer Share = {best_rpc_region['Customer_Share']:.2%} | "
    f"Revenue Share = {best_rpc_region['Revenue_Share']:.2%} | "
    f"Revenue per Customer = {best_rpc_region['Revenue_per_Customer']:,.0f}"
)
print(
    f"Best repeat region: {best_repeat_region['region']} | "
    f"Repeat Purchase Rate = {best_repeat_region['Repeat_Purchase_Rate']:.2%}"
)
print(
    f"Highest refund-risk region: {highest_refund_region['region']} | "
    f"Refund / Revenue = {highest_refund_region['Refund_to_Revenue']:.2%}"
)

In [ ]:
# ============================================================
# G0. CONFIG & IMPORTS
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, r2_score
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

DATA_DIR = "."

SALES_PATH = os.path.join(DATA_DIR, "sales.csv")
SAMPLE_PATH = os.path.join(DATA_DIR, "sample_submission.csv")

OUTPUT_PATH = "forecast_submission.csv"

In [ ]:
# ============================================================
# G1. LOAD DATA
# ============================================================

sales = pd.read_csv(SALES_PATH)

if "Date" in sales.columns:
    sales = sales.rename(columns={"Date": "date"})
elif "date" not in sales.columns:
    raise ValueError("sales.csv must contain Date or date column.")

sales["date"] = pd.to_datetime(sales["date"])
sales = sales.sort_values("date").reset_index(drop=True)

required_cols = ["date", "Revenue", "COGS"]
missing_cols = [c for c in required_cols if c not in sales.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in sales.csv: {missing_cols}")

sample_submission = pd.read_csv(SAMPLE_PATH)

if "Date" in sample_submission.columns:
    future_dates = pd.to_datetime(sample_submission["Date"])
elif "date" in sample_submission.columns:
    future_dates = pd.to_datetime(sample_submission["date"])
else:
    raise ValueError("sample_submission.csv must contain Date or date column.")

future_dates = (
    pd.Series(future_dates)
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

forecast_horizon = len(future_dates)

print("Train period:", sales["date"].min(), "to", sales["date"].max())
print("Forecast period:", future_dates.min(), "to", future_dates.max())
print("Forecast horizon:", forecast_horizon)

In [ ]:
# ============================================================
# G2. METRICS
# ============================================================

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def calc_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

In [ ]:
# ============================================================
# G3. MODEL 1 - RECURSIVE SEASONAL NAIVE
# ============================================================

def forecast_recursive_seasonal_naive(train_df, future_dates, target_col):
    """
    Dự báo bằng cùng ngày năm trước.
    Nếu dự báo sang năm thứ hai, dùng chính forecast năm trước làm input.
    Không dùng actual future.
    """

    history = dict(zip(train_df["date"], train_df[target_col]))
    preds = []

    for current_date in pd.to_datetime(future_dates):
        previous_year_date = current_date - pd.DateOffset(years=1)

        if previous_year_date in history:
            pred = history[previous_year_date]
        else:
            fallback_date = current_date - pd.Timedelta(days=365)
            pred = history.get(
                fallback_date,
                train_df[target_col].tail(365).median()
            )

        pred = max(float(pred), 0)
        preds.append(pred)

        history[current_date] = pred

    return np.asarray(preds)

In [ ]:
# ============================================================
# G4. MODEL 2 - DAMPED YOY SEASONAL NAIVE
# ============================================================

def get_recent_damped_growth(train_df, target_col, clip=0.12, damp=0.50):
    """
    Tính tăng trưởng YoY gần đây, có cap và damp để tránh forecast quá cực đoan.
    Dữ liệu sau 2018 đã đổi regime nên không để trend quá mạnh.
    """

    yearly = (
        train_df
        .assign(year=train_df["date"].dt.year)
        .groupby("year")[target_col]
        .sum()
        .sort_index()
    )

    growth = yearly.pct_change().dropna()

    if len(growth) == 0:
        return 0.0

    recent_growth = growth.tail(3)

    weights = np.array([0.20, 0.30, 0.50])[-len(recent_growth):]
    g = np.average(recent_growth.values, weights=weights)

    g = np.clip(g, -clip, clip)
    g = g * damp

    return float(g)


def forecast_damped_yoy_seasonal(train_df, future_dates, target_col):
    base_pred = forecast_recursive_seasonal_naive(
        train_df=train_df,
        future_dates=future_dates,
        target_col=target_col
    )

    g = get_recent_damped_growth(train_df, target_col)

    last_train_year = train_df["date"].dt.year.max()

    factors = []

    for current_date in pd.to_datetime(future_dates):
        step = max(current_date.year - last_train_year, 1)
        factors.append((1 + g) ** step)

    pred = base_pred * np.asarray(factors)

    return np.maximum(pred, 0)

In [ ]:
# ============================================================
# G5. MODEL 3 - RECENT DAY-OF-YEAR SEASONAL PROFILE
# ============================================================

def forecast_recent_seasonal_profile(
    train_df,
    future_dates,
    target_col,
    n_recent_years=4,
    growth_clip=0.12,
    growth_damp=0.50,
    recency_decay=0.70
):
    """
    Mô hình profile mùa vụ:
    - Dùng tỷ trọng doanh thu theo ngày trong năm từ các năm gần nhất
    - Dự báo annual level bằng recent damped growth
    - Phù hợp khi dữ liệu có mùa vụ mạnh nhưng structural break sau 2018
    """

    df = train_df.copy()
    df["year"] = df["date"].dt.year
    df["doy"] = df["date"].dt.dayofyear.clip(upper=365)

    year_counts = df.groupby("year").size()
    complete_years = year_counts[year_counts >= 360].index.tolist()

    if len(complete_years) == 0:
        complete_years = sorted(df["year"].unique().tolist())

    recent_years = complete_years[-n_recent_years:]

    sub = df[df["year"].isin(recent_years)].copy()

    annual_total = sub.groupby("year")[target_col].sum()
    sub["annual_total"] = sub["year"].map(annual_total)
    sub["daily_share"] = sub[target_col] / sub["annual_total"]

    year_rank = {year: i for i, year in enumerate(recent_years)}
    max_rank = len(recent_years) - 1

    sub["weight"] = sub["year"].map(
        lambda y: recency_decay ** (max_rank - year_rank[y])
    )

    weighted_profile = (
        sub
        .assign(weighted_share=sub["daily_share"] * sub["weight"])
        .groupby("doy", as_index=False)
        .agg(
            weighted_share=("weighted_share", "sum"),
            weight=("weight", "sum")
        )
    )

    weighted_profile["share"] = (
        weighted_profile["weighted_share"] / weighted_profile["weight"]
    )

    profile = (
        pd.DataFrame({"doy": np.arange(1, 366)})
        .merge(weighted_profile[["doy", "share"]], on="doy", how="left")
    )

    profile["share"] = (
        profile["share"]
        .interpolate()
        .bfill()
        .ffill()
    )

    profile["share"] = (
        profile["share"]
        .rolling(window=7, center=True, min_periods=1)
        .mean()
    )

    profile["share"] = profile["share"] / profile["share"].sum()

    share_map = dict(zip(profile["doy"], profile["share"]))

    last_year = df["year"].max()
    last_annual_total = df[df["year"] == last_year][target_col].sum()

    g = get_recent_damped_growth(
        train_df=df,
        target_col=target_col,
        clip=growth_clip,
        damp=growth_damp
    )

    annual_forecast = {}

    for year in sorted(pd.to_datetime(future_dates).dt.year.unique()):
        step = max(year - last_year, 1)
        annual_forecast[year] = last_annual_total * ((1 + g) ** step)

    preds = []

    for current_date in pd.to_datetime(future_dates):
        doy = min(current_date.dayofyear, 365)
        pred = annual_forecast[current_date.year] * share_map[doy]
        preds.append(pred)

    return np.maximum(np.asarray(preds), 0)

In [ ]:
# ============================================================
# G6. MODEL 4 - ETS LOG-DAMPED, RECENT REGIME
# ============================================================

def forecast_ets_log_damped(
    train_df,
    future_dates,
    target_col,
    recent_start="2019-01-01"
):
    """
    ETS trên log1p target.
    Chỉ ưu tiên recent regime sau 2019 để tránh bị kéo bởi giai đoạn 2012-2018.
    """

    fit_df = train_df[train_df["date"] >= pd.Timestamp(recent_start)].copy()

    if len(fit_df) < 730:
        fit_df = train_df.copy()

    y = np.log1p(fit_df[target_col].astype(float).values)

    try:
        model = ExponentialSmoothing(
            y,
            trend="add",
            damped_trend=True,
            seasonal="add",
            seasonal_periods=365,
            initialization_method="estimated"
        )

        fitted = model.fit(
            optimized=True,
            use_brute=False
        )

        pred = np.expm1(fitted.forecast(len(future_dates)))

    except Exception:
        model = ExponentialSmoothing(
            y,
            trend="add",
            damped_trend=True,
            seasonal=None,
            initialization_method="estimated"
        )

        fitted = model.fit(
            optimized=True,
            use_brute=False
        )

        pred = np.expm1(fitted.forecast(len(future_dates)))

    return np.maximum(np.asarray(pred), 0)

In [ ]:
# ============================================================
# G7. COLLECT BASE MODEL PREDICTIONS
# ============================================================

def forecast_base_models(train_df, future_dates, target_col):
    preds = {}

    preds["seasonal_naive"] = forecast_recursive_seasonal_naive(
        train_df=train_df,
        future_dates=future_dates,
        target_col=target_col
    )

    preds["damped_yoy_seasonal"] = forecast_damped_yoy_seasonal(
        train_df=train_df,
        future_dates=future_dates,
        target_col=target_col
    )

    preds["recent_profile"] = forecast_recent_seasonal_profile(
        train_df=train_df,
        future_dates=future_dates,
        target_col=target_col,
        n_recent_years=4,
        growth_clip=0.12,
        growth_damp=0.50,
        recency_decay=0.70
    )

    preds["ets_log_damped_recent"] = forecast_ets_log_damped(
        train_df=train_df,
        future_dates=future_dates,
        target_col=target_col,
        recent_start="2019-01-01"
    )

    return preds

In [ ]:
# ============================================================
# G8. ROLLING 548-DAY BACKTEST
# ============================================================

def make_validation_starts(sales_df, horizon):
    """
    Tạo các fold giống hidden test:
    mỗi fold forecast liên tục horizon ngày.
    """

    candidate_starts = [
        "2018-07-01",
        "2019-07-01",
        "2020-07-01",
        "2021-07-01"
    ]

    max_date = sales_df["date"].max()
    valid_starts = []

    for s in candidate_starts:
        start = pd.Timestamp(s)
        end = start + pd.Timedelta(days=horizon - 1)

        if end <= max_date:
            valid_starts.append(start)

    if len(valid_starts) == 0:
        raise ValueError("No valid backtest fold can be created.")

    return valid_starts


def generate_weight_grid(n_models, step=0.05):
    """
    Sinh grid trọng số ensemble.
    Tổng trọng số = 1.
    """

    if n_models == 1:
        return [np.array([1.0])]

    values = np.arange(0, 1 + 1e-9, step)
    grids = []

    def backtrack(current, remaining, depth):
        if depth == n_models - 1:
            grids.append(np.array(current + [remaining]))
            return

        for v in values:
            if v <= remaining + 1e-9:
                backtrack(current + [v], remaining - v, depth + 1)

    backtrack([], 1.0, 0)

    return grids


def rolling_backtest_target(sales_df, target_col, horizon):
    valid_starts = make_validation_starts(sales_df, horizon)

    oof_parts = []
    metric_rows = []

    model_names = None

    for valid_start in valid_starts:
        valid_end = valid_start + pd.Timedelta(days=horizon - 1)

        train_df = sales_df[sales_df["date"] < valid_start].copy()

        valid_df = sales_df[
            (sales_df["date"] >= valid_start) &
            (sales_df["date"] <= valid_end)
        ].copy()

        base_preds = forecast_base_models(
            train_df=train_df,
            future_dates=valid_df["date"],
            target_col=target_col
        )

        if model_names is None:
            model_names = list(base_preds.keys())

        fold_oof = valid_df[["date", target_col]].copy()
        fold_oof["fold_start"] = valid_start

        for model_name in model_names:
            pred = base_preds[model_name]
            fold_oof[model_name] = pred

            m = calc_metrics(valid_df[target_col], pred)

            metric_rows.append({
                "Target": target_col,
                "Fold_Start": valid_start,
                "Model": model_name,
                **m
            })

        oof_parts.append(fold_oof)

    oof = pd.concat(oof_parts, ignore_index=True)

    best = None

    for weights in generate_weight_grid(len(model_names), step=0.05):
        pred = np.zeros(len(oof))

        for i, model_name in enumerate(model_names):
            pred += weights[i] * oof[model_name].values

        m = calc_metrics(oof[target_col], pred)

        result = {
            "Target": target_col,
            "Model": "weighted_ensemble",
            "Weights": weights,
            **m
        }

        if best is None or result["MAE"] < best["MAE"]:
            best = result

    oof["ensemble_pred"] = 0.0

    for i, model_name in enumerate(model_names):
        oof["ensemble_pred"] += best["Weights"][i] * oof[model_name]

    m_ens = calc_metrics(oof[target_col], oof["ensemble_pred"])

    metric_rows.append({
        "Target": target_col,
        "Fold_Start": "OOF_ALL",
        "Model": "weighted_ensemble",
        **m_ens
    })

    metrics = pd.DataFrame(metric_rows)

    return metrics, oof, model_names, best

In [ ]:
# ============================================================
# G9. RUN BACKTEST FOR REVENUE AND COGS
# ============================================================

revenue_metrics, revenue_oof, revenue_model_names, revenue_best = rolling_backtest_target(
    sales_df=sales,
    target_col="Revenue",
    horizon=forecast_horizon
)

cogs_metrics, cogs_oof, cogs_model_names, cogs_best = rolling_backtest_target(
    sales_df=sales,
    target_col="COGS",
    horizon=forecast_horizon
)

all_metrics = pd.concat([revenue_metrics, cogs_metrics], ignore_index=True)

display(
    all_metrics
    .sort_values(["Target", "Model", "Fold_Start"])
)

summary_metrics = (
    all_metrics[all_metrics["Fold_Start"] == "OOF_ALL"]
    .sort_values(["Target", "MAE"])
)

display(summary_metrics)

print("Revenue base models:", revenue_model_names)
print("Revenue best weights:", revenue_best)

print("COGS base models:", cogs_model_names)
print("COGS best weights:", cogs_best)

In [ ]:
# ============================================================
# G10. MONTHLY BIAS CORRECTION FROM BACKTEST ONLY
# ============================================================

def learn_monthly_bias_correction(oof_df, target_col, pred_col="ensemble_pred"):
    """
    Học correction theo calendar month từ validation quá khứ.
    Không dùng sample_submission Revenue/COGS.
    """

    temp = oof_df.copy()
    temp["calendar_month"] = temp["date"].dt.month

    temp["ratio"] = temp[target_col] / temp[pred_col].replace(0, np.nan)

    correction = (
        temp
        .groupby("calendar_month", as_index=False)
        .agg(raw_ratio=("ratio", "median"))
    )

    shrink = 0.50

    correction["correction_factor"] = (
        1 + shrink * (correction["raw_ratio"] - 1)
    )

    correction["correction_factor"] = correction["correction_factor"].clip(0.85, 1.15)

    return correction[["calendar_month", "correction_factor"]]


revenue_monthly_correction = learn_monthly_bias_correction(
    revenue_oof,
    target_col="Revenue",
    pred_col="ensemble_pred"
)

cogs_monthly_correction = learn_monthly_bias_correction(
    cogs_oof,
    target_col="COGS",
    pred_col="ensemble_pred"
)

display(revenue_monthly_correction)
display(cogs_monthly_correction)

In [ ]:
# ============================================================
# G11. REFIT FULL TRAIN AND FORECAST FUTURE
# ============================================================

def ensemble_forecast_full_train(sales_df, future_dates, target_col, model_names, best_result):
    base_preds = forecast_base_models(
        train_df=sales_df,
        future_dates=future_dates,
        target_col=target_col
    )

    pred = np.zeros(len(future_dates))

    for i, model_name in enumerate(model_names):
        pred += best_result["Weights"][i] * base_preds[model_name]

    return np.maximum(pred, 0), base_preds


forecast_revenue_raw, revenue_base_future = ensemble_forecast_full_train(
    sales_df=sales,
    future_dates=future_dates,
    target_col="Revenue",
    model_names=revenue_model_names,
    best_result=revenue_best
)

forecast_cogs_direct_raw, cogs_base_future = ensemble_forecast_full_train(
    sales_df=sales,
    future_dates=future_dates,
    target_col="COGS",
    model_names=cogs_model_names,
    best_result=cogs_best
)

future_forecast = pd.DataFrame({
    "Date": future_dates,
    "Forecast_Revenue_Raw": forecast_revenue_raw,
    "Forecast_COGS_Direct_Raw": forecast_cogs_direct_raw
})

future_forecast["calendar_month"] = future_forecast["Date"].dt.month

future_forecast = future_forecast.merge(
    revenue_monthly_correction.rename(
        columns={"correction_factor": "revenue_correction_factor"}
    ),
    on="calendar_month",
    how="left"
)

future_forecast = future_forecast.merge(
    cogs_monthly_correction.rename(
        columns={"correction_factor": "cogs_correction_factor"}
    ),
    on="calendar_month",
    how="left"
)

future_forecast["revenue_correction_factor"] = (
    future_forecast["revenue_correction_factor"].fillna(1.0)
)

future_forecast["cogs_correction_factor"] = (
    future_forecast["cogs_correction_factor"].fillna(1.0)
)

future_forecast["Forecast_Revenue"] = (
    future_forecast["Forecast_Revenue_Raw"] *
    future_forecast["revenue_correction_factor"]
)

future_forecast["Forecast_COGS_Direct"] = (
    future_forecast["Forecast_COGS_Direct_Raw"] *
    future_forecast["cogs_correction_factor"]
)

In [ ]:
# ============================================================
# G12. COGS RATIO MODEL AND BLENDING
# ============================================================

def build_recent_monthly_cogs_rate(sales_df, recent_start="2019-01-01"):
    """
    Tính tỷ lệ COGS / Revenue theo tháng từ recent regime.
    Dùng để tạo COGS theo logic biên lợi nhuận.
    """

    recent = sales_df[sales_df["date"] >= pd.Timestamp(recent_start)].copy()

    if len(recent) < 365:
        recent = sales_df.copy()

    recent["calendar_month"] = recent["date"].dt.month

    monthly_rate = (
        recent
        .groupby("calendar_month", as_index=False)
        .agg(
            Revenue=("Revenue", "sum"),
            COGS=("COGS", "sum")
        )
    )

    monthly_rate["COGS_Rate"] = monthly_rate["COGS"] / monthly_rate["Revenue"]

    return monthly_rate[["calendar_month", "COGS_Rate"]]


monthly_cogs_rate = build_recent_monthly_cogs_rate(
    sales_df=sales,
    recent_start="2019-01-01"
)

future_forecast = future_forecast.merge(
    monthly_cogs_rate,
    on="calendar_month",
    how="left"
)

global_cogs_rate = sales.loc[sales["date"] >= "2019-01-01", "COGS"].sum() / \
                   sales.loc[sales["date"] >= "2019-01-01", "Revenue"].sum()

future_forecast["COGS_Rate"] = future_forecast["COGS_Rate"].fillna(global_cogs_rate)

future_forecast["Forecast_COGS_Ratio"] = (
    future_forecast["Forecast_Revenue"] *
    future_forecast["COGS_Rate"]
)

In [ ]:
# ============================================================
# G13. VALIDATE COGS BLENDING WEIGHT
# ============================================================

def validate_cogs_blend(sales_df, horizon, revenue_oof, cogs_oof):
    """
    Tìm blend tốt nhất giữa:
    - COGS direct forecast
    - COGS = predicted Revenue × monthly COGS rate
    trên OOF validation.
    """

    oof = cogs_oof[["date", "COGS", "ensemble_pred"]].copy()
    oof = oof.rename(columns={"ensemble_pred": "cogs_direct_pred"})

    rev = revenue_oof[["date", "ensemble_pred"]].copy()
    rev = rev.rename(columns={"ensemble_pred": "revenue_pred"})

    oof = oof.merge(rev, on="date", how="left")

    rate = build_recent_monthly_cogs_rate(
        sales_df=sales_df,
        recent_start="2019-01-01"
    )

    oof["calendar_month"] = oof["date"].dt.month

    oof = oof.merge(rate, on="calendar_month", how="left")

    global_rate = sales_df.loc[sales_df["date"] >= "2019-01-01", "COGS"].sum() / \
                  sales_df.loc[sales_df["date"] >= "2019-01-01", "Revenue"].sum()

    oof["COGS_Rate"] = oof["COGS_Rate"].fillna(global_rate)

    oof["cogs_ratio_pred"] = oof["revenue_pred"] * oof["COGS_Rate"]

    best = None

    for w_direct in np.arange(0, 1.01, 0.05):
        pred = (
            w_direct * oof["cogs_direct_pred"] +
            (1 - w_direct) * oof["cogs_ratio_pred"]
        )

        m = calc_metrics(oof["COGS"], pred)

        result = {
            "w_direct": w_direct,
            "w_ratio": 1 - w_direct,
            **m
        }

        if best is None or result["MAE"] < best["MAE"]:
            best = result

    return best


cogs_blend_best = validate_cogs_blend(
    sales_df=sales,
    horizon=forecast_horizon,
    revenue_oof=revenue_oof,
    cogs_oof=cogs_oof
)

print("Best COGS blend:", cogs_blend_best)

future_forecast["Forecast_COGS"] = (
    cogs_blend_best["w_direct"] * future_forecast["Forecast_COGS_Direct"] +
    cogs_blend_best["w_ratio"] * future_forecast["Forecast_COGS_Ratio"]
)

future_forecast["Forecast_Revenue"] = future_forecast["Forecast_Revenue"].clip(lower=0)
future_forecast["Forecast_COGS"] = future_forecast["Forecast_COGS"].clip(lower=0)

In [ ]:
# ============================================================
# G14. BUSINESS SANITY CHECK
# ============================================================

future_forecast["Forecast_Gross_Margin_Pct"] = (
    (future_forecast["Forecast_Revenue"] - future_forecast["Forecast_COGS"]) /
    future_forecast["Forecast_Revenue"].replace(0, np.nan)
)

future_monthly_forecast = (
    future_forecast
    .assign(period=future_forecast["Date"].dt.to_period("M").astype(str))
    .groupby("period", as_index=False)
    .agg(
        Forecast_Revenue=("Forecast_Revenue", "sum"),
        Forecast_COGS=("Forecast_COGS", "sum")
    )
)

future_monthly_forecast["Forecast_Gross_Margin_Pct"] = (
    (future_monthly_forecast["Forecast_Revenue"] - future_monthly_forecast["Forecast_COGS"]) /
    future_monthly_forecast["Forecast_Revenue"].replace(0, np.nan)
)

display(future_forecast.head())
display(future_monthly_forecast.head(18))

print("Forecast Revenue total:", future_forecast["Forecast_Revenue"].sum())
print("Forecast COGS total:", future_forecast["Forecast_COGS"].sum())
print("Average forecast GM%:", future_forecast["Forecast_Gross_Margin_Pct"].mean())

In [ ]:
# ============================================================
# G15. EXPORT SUBMISSION
# ============================================================

forecast_submission = pd.DataFrame({
    "Date": future_forecast["Date"],
    "Revenue": future_forecast["Forecast_Revenue"],
    "COGS": future_forecast["Forecast_COGS"]
})

forecast_submission.to_csv(OUTPUT_PATH, index=False)

display(forecast_submission.head())
print(f"Saved file: {OUTPUT_PATH}")

files.download(OUTPUT_PATH)